<div style="text-align: center; padding: 40px 0; border-bottom: 2px solid #e0e0e0; margin-bottom: 20px;">
  <h1 style="font-size: 2.5em; font-weight: 700; color: #000000; margin: 0;">
    🇫🇮 Finland Public Transport
  </h1>
  <h2 style="font-size: 1.4em; font-weight: 300; color: #555; margin: 10px 0 0 0;">
    GTFS and NeTEx Dataset Analysis — 2026
  </h2>
  <p style="color: #999; margin-top: 12px; font-size: 0.9em;">
    Source: mobility.mobility-database.fintraffic.fi · Provider: Fintraffic (Finnish National Access Point) · Format: GTFS & NeTEx
  </p>
</div>

## Data Source

Both datasets were obtained from Fintraffic's **Koontipalvelu** (national mobility
database), which aggregates public transport data from all operators in Finland.
The download page is:

`https://mobility.mobility-database.fintraffic.fi/en`

No registration or API key is required. Both feeds are free under **CC BY 4.0**.

**GTFS Finland**
- National feed aggregating all public transport operators in Finland
- Downloaded as a single static ZIP file from the Koontipalvelu page above

**NeTEx Finland**
- National NeTEx feed published through the same platform
- Downloaded as a single static ZIP file from the Koontipalvelu page above

**Important note on the NeTEx feed**

Unlike Norway and Sweden, where operators produce NeTEx data directly, most
operators in Finland still work natively in GTFS. Fintraffic's validation and
conversion tool **VACO** automatically converts their GTFS data into NeTEx for
operators who do not produce it themselves. This means the Finnish NeTEx feed
is largely a converted version of the GTFS data rather than an independently
produced dataset.

This is confirmed by Finland's EU ITS Progress Report (2023), which states:

"*The encountered challenge is that the transport service industry in Finland
has not built its systems on NeTEx and SIRI standards but is relying more on
open GTFS and GTFS-RT standards.*"

Source: Finland ITS Progress Report 2023, European Commission:
`https://transport.ec.europa.eu/document/download/7a90d4d2-1fa9-4635-95b2-8f67e46ce45d_en?filename=2024_fi_its_progress_report_2023.pdf`

Finland is still transitioning to NeTEx and is targeting the use of the Nordic
NeTEx profile developed by Entur in Norway. This means the comparison results
for Finland may show artificially high match rates. This is not because the two formats
are independently consistent, but because the NeTEx data was largely derived
from GTFS in the first place. This will be noted when interpreting the results.

## Table of Contents

**Data Source**

**Part 1: GTFS Exploration**
- Inspect GTFS stop structure
- Inspect GTFS stop hierarchy
- Recheck parent_station after cleaning IDs
- Prepare GTFS station-level table
- Inspect GTFS route labels
- Inspect routes without route_short_name
- Prepare GTFS line-level table
- Deduplicate GTFS lines by public label
- Inspect GTFS transport modes
- Inspect GTFS service_id coverage
- Inspect GTFS calendar date ranges and exceptions
- Inspect GTFS feed information
- Classify GTFS calendar service IDs
- Inspect long GTFS calendar validity periods
- Rebuild GTFS active service dates
- Check active-date coverage for trip service IDs
- Inspect trip service IDs without active dates
- Explore an example
- Prepare GTFS calendar summary

**Part 2: NeTEx Exploration**
- Inspect NeTEx ZIP structure
- NeTEx structure check: element counter
- Inspect sample NeTEx StopPlace records
- Check StopPlace coordinate coverage
- Extract all NeTEx StopPlaces
- Check whether Finnish NeTEx StopPlaces have a parent-child structure
- Note on stop-level comparison approach for Finland
- Prepare GTFS stop-level table for Finland
- Inspect sample NeTEx Lines
- Extract all NeTEx lines
- Inspect duplicate PublicCodes
- Prepare NeTEx deduplicated line table
- Inspect NeTEx lines without PublicCode
- Inspect NeTEx transport modes
- Inspect NeTEx calendar references in line files
- Inspect sample ServiceJourney DayTypeRefs
- Search calendar keywords in shared data
- Inspect sample DayType and OperatingPeriod elements
- Inspect internal structure of DayType and OperatingPeriod
- Inspect OperatingDay structure
- Extract sample OperatingDays and OperatingPeriods carefully
- Inspect DayTypes used by ServiceJourneys
- Check whether DayTypes are linked to OperatingPeriods by ID
- Conclusion: NeTEx calendar structure for Finland

**Part 3: GTFS–NeTEx Comparison**
- 3.1 Stop-level comparison for Finland
  - Inspect GTFS stops not matched by coordinates
  - Nearest-neighbour distance check for unmatched GTFS stops
  - Check names for nearest NeTEx matches
  - Final stop-level comparison summary
- 3.2 Line-level comparison for Finland
  - Inspect unmatched line labels
  - Conclusion: line-level comparison for Finland
- 3.3 Calendar comparison for Finland

# Part 1: GTFS Exploration

All third-party and standard-library imports used in this notebook are consolidated here.

In [ ]:
from pathlib import Path
import zipfile
import pandas as pd
from collections import defaultdict
import xml.etree.ElementTree as ET
from sklearn.neighbors import BallTree
import numpy as np
import re
import unicodedata

In [1]:
# Set path

data_dir = Path("data")

gtfs_zip_path = data_dir / "finland_gtfs.zip"

print("GTFS exists:", gtfs_zip_path.exists(), gtfs_zip_path)

GTFS exists: True data/finland_gtfs.zip


In [2]:
# Inspect GTFS ZIP contents
with zipfile.ZipFile(gtfs_zip_path, "r") as z:
    gtfs_files = pd.DataFrame([
        {
            "name":    info.filename,
            "size_mb": round(info.file_size / (1024 * 1024), 2)
        }
        for info in z.infolist()
    ])

display(gtfs_files.sort_values("size_mb", ascending=False))

,name,size_mb
11,stop_times.txt,1832.31
9,shapes.txt,1201.65
15,trips.txt,68.77
14,translations.txt,66.61
3,calendar_dates.txt,26.52
12,stops.txt,7.01
2,calendar.txt,1.36
8,routes.txt,0.46
13,transfers.txt,0.27
10,stop_areas.txt,0.16


In [3]:
# Helper function to read GTFS tables from the ZIP

def read_gtfs_from_zip(zip_path: Path, filename: str) -> pd.DataFrame:
    with zipfile.ZipFile(zip_path, "r") as z:
        names = z.namelist()

        # If the exact file name is present, use it directly
        if filename in names:
            target = filename
        else:
            # Otherwise, search for it anywhere inside the ZIP
            matches = [n for n in names if n.lower().endswith("/" + filename.lower())]

            if not matches:
                raise FileNotFoundError(f"{filename} not found in ZIP. Example entries: {names[:20]}")
            if len(matches) > 1:
                raise FileNotFoundError(f"Multiple matches found for {filename}: {matches}")

            target = matches[0]

        with z.open(target) as f:
            return pd.read_csv(f, low_memory=False)

In [4]:
# Load core GTFS tables
stops = read_gtfs_from_zip(gtfs_zip_path, "stops.txt")
routes = read_gtfs_from_zip(gtfs_zip_path, "routes.txt")
trips = read_gtfs_from_zip(gtfs_zip_path, "trips.txt")
calendar = read_gtfs_from_zip(gtfs_zip_path, "calendar.txt")
calendar_dates = read_gtfs_from_zip(gtfs_zip_path, "calendar_dates.txt")

print("stops:", stops.shape)
print("routes:", routes.shape)
print("trips:", trips.shape)
print("calendar:", calendar.shape)
print("calendar_dates:", calendar_dates.shape)

stops: (89547, 17)
routes: (7226, 13)
trips: (660650, 10)
calendar: (25226, 10)
calendar_dates: (817610, 3)


In [5]:
print("stops columns:")
print(list(stops.columns))
display(stops.head())

print("routes columns:")
print(list(routes.columns))
display(routes.head())

print("trips columns:")
print(list(trips.columns))
display(trips.head())

print("calendar columns:")
print(list(calendar.columns))
display(calendar.head())

print("calendar_dates columns:")
print(list(calendar_dates.columns))
display(calendar_dates.head())

stops columns:
['stop_id', 'stop_code', 'stop_name', 'tts_stop_name', 'stop_desc', 'stop_lat', 'stop_lon', 'zone_id', 'stop_url', 'location_type', 'parent_station', 'stop_timezone', 'wheelchair_boarding', 'level_id', 'platform_code', 'digiroad_id', 'vehicle_type']


,stop_id,stop_code,stop_name,tts_stop_name,stop_desc,stop_lat,stop_lon,zone_id,stop_url,location_type,parent_station,stop_timezone,wheelchair_boarding,level_id,platform_code,digiroad_id,vehicle_type
0,300000,NaN,Kamppi (lähiliikenneterminaali),NaN,NaN,60.169008,24.931662,A,NaN,1,NaN,Europe/Helsinki,0.0,NaN,NaN,NaN,NaN
1,300001,NaN,Elielinaukio,NaN,NaN,60.171808,24.939488,A,NaN,1,NaN,Europe/Helsinki,0.0,NaN,NaN,NaN,NaN
2,300002,NaN,Rautatientori,NaN,NaN,60.171283,24.942572,A,NaN,1,NaN,Europe/Helsinki,0.0,NaN,NaN,NaN,NaN
3,300003,NaN,Pasila,NaN,NaN,60.198118,24.934074,A,NaN,1,NaN,Europe/Helsinki,0.0,NaN,NaN,NaN,NaN
4,300004,NaN,Malmin asema (terminaali),NaN,NaN,60.250292,25.009293,B,NaN,1,NaN,Europe/Helsinki,0.0,NaN,NaN,NaN,NaN


routes columns:
['route_id', 'agency_id', 'route_short_name', 'route_long_name', 'route_desc', 'route_type', 'route_url', 'route_color', 'route_text_color', 'route_sort_order', 'continuous_pickup', 'continuous_drop_off', 'network_id']


,route_id,agency_id,route_short_name,route_long_name,route_desc,route_type,route_url,route_color,route_text_color,route_sort_order,continuous_pickup,continuous_drop_off,network_id
0,20000,1000,1,Eira - Kamppi (M) - Töölö - Sörnäinen (M) - Kä...,NaN,0,http://aikataulut.hsl.fi/linjat/fi/h1_1a.html,NaN,NaN,NaN,NaN,NaN,NaN
1,20001,1000,10,Ullanlinna - Lasipalatsi - Ruskeasuo - Pikku H...,NaN,0,http://aikataulut.hsl.fi/linjat/fi/h10.html,NaN,NaN,NaN,NaN,NaN,NaN
2,20002,1000,105,Lauttasaari(M)-Westend-Haukilahti-Mankkaa,NaN,701,http://aikataulut.hsl.fi/linjat/fi/b105.html,NaN,NaN,NaN,NaN,NaN,NaN
3,20004,1000,10H,Pikku Huopalahti - Ruskeasuo - Ooppera,NaN,0,http://aikataulut.hsl.fi/linjat/fi/h10H.html,NaN,NaN,NaN,NaN,NaN,NaN
4,20005,1000,111,Otaniemi-Tapiola (M)-Westend-Matinkylä (M),NaN,701,http://aikataulut.hsl.fi/linjat/fi/b111.html,NaN,NaN,NaN,NaN,NaN,NaN


trips columns:
['trip_id', 'route_id', 'service_id', 'trip_headsign', 'trip_short_name', 'direction_id', 'block_id', 'shape_id', 'wheelchair_accessible', 'bikes_allowed']


,trip_id,route_id,service_id,trip_headsign,trip_short_name,direction_id,block_id,shape_id,wheelchair_accessible,bikes_allowed
0,12578_T2026-BR-VS-M-P_1273_0_062500_065000_0,100935,12578_T2026-BR-VS-M-P,Lemu,NaN,0.0,NaN,12578_65,0.0,0.0
1,12578_T2026-BR-VS-M-P_1273_1_171000_173500_0,100935,12578_T2026-BR-VS-M-P,Mietoinen,NaN,1.0,NaN,12578_68,0.0,0.0
2,11409_235062.7010,101195,11409_235062.7010,Sairaala,NaN,NaN,NaN,11409_16,NaN,NaN
3,11409_235063.7010,101195,11409_235063.7010,Sairaala,NaN,NaN,NaN,11409_16,NaN,NaN
4,11409_235064.7010,101195,11409_235064.7010,Sairaala,NaN,NaN,NaN,11409_16,NaN,NaN


calendar columns:
['service_id', 'monday', 'tuesday', 'wednesday', 'thursday', 'friday', 'saturday', 'sunday', 'start_date', 'end_date']


,service_id,monday,tuesday,wednesday,thursday,friday,saturday,sunday,start_date,end_date
0,10040_Livi_KOULPV_M_P_Koski_Tl,1,1,1,1,1,0,0,20140701,20280807
1,10040_Livi_KOULPV_M_P_Loimaa,1,1,1,1,1,0,0,20140701,20280807
2,10040_Livi_KOULPV_M_P_Pöytyä,1,1,1,1,1,0,0,20140701,20280807
3,10040_Livi_KOULPV_P_Loimaa,0,0,0,0,1,0,0,20140701,20280807
4,10058_Livi_KOULPV_M_P_Koski_Tl,1,1,1,1,1,0,0,20140701,20280807


calendar_dates columns:
['service_id', 'date', 'exception_type']


,service_id,date,exception_type
0,10215_FSR:DayType:f516a7a1-97bb-41b6-a43d-b6a9...,20150601,1
1,10215_FSR:DayType:f516a7a1-97bb-41b6-a43d-b6a9...,20150602,1
2,10215_FSR:DayType:f516a7a1-97bb-41b6-a43d-b6a9...,20150603,1
3,10215_FSR:DayType:f516a7a1-97bb-41b6-a43d-b6a9...,20150604,1
4,10215_FSR:DayType:f516a7a1-97bb-41b6-a43d-b6a9...,20150605,1


## Comment

The `stops` table contains a clear stop hierarchy because it includes `location_type` and `parent_station`.  
The first rows are station-level stops with `location_type = 1`.

The `routes` table contains `route_short_name`, which can be used for the line-level comparison.  


The `calendar` and `calendar_dates` tables are both present.  
This means the GTFS validity must be rebuilt by combining weekly service rules with calendar exceptions.

## Inspect GTFS stop hierarchy

The GTFS `stops.txt` file contains `location_type` and `parent_station`. I first check how the stop hierarchy is structured.

In [6]:
# Inspect location_type distribution
location_type_counts = (
    stops["location_type"]
    .fillna("missing")
    .value_counts(dropna=False)
    .reset_index()
)

location_type_counts.columns = ["location_type", "count"]

location_type_counts

,location_type,count
0,0,88750
1,1,797


In [7]:
# Check parent_station structure

stop_ids = set(stops["stop_id"].astype(str))

parent_station_values = (
    stops["parent_station"]
    .dropna()
    .astype(str)
)

unique_parent_stations = set(parent_station_values)

parent_station_check = pd.DataFrame({
    "metric": [
        "total stop rows",
        "rows with parent_station",
        "unique parent_station values",
        "parent_station values found as stop_id",
        "parent_station values not found as stop_id"
    ],
    "value": [
        len(stops),
        stops["parent_station"].notna().sum(),
        len(unique_parent_stations),
        len(unique_parent_stations & stop_ids),
        len(unique_parent_stations - stop_ids)
    ]
})

parent_station_check

,metric,value
0,total stop rows,89547
1,rows with parent_station,1556
2,unique parent_station values,658
3,parent_station values found as stop_id,0
4,parent_station values not found as stop_id,658


## Comment

Most GTFS stop rows are regular stop/platform rows with `location_type = 0`.  
There are also 797 station-level rows with `location_type = 1`.

This means Finland does have a GTFS stop hierarchy, but it is much smaller than for example Norway.  
Only 1,556 rows have a `parent_station`.

The parent-station check shows that none of the `parent_station` values were found as `stop_id`.  


## Recheck parent_station after cleaning IDs

The first parent-station check showed that no `parent_station` values were found as `stop_id`.

I clean the IDs and repeat the check.  
This avoids false mismatches caused by formatting differences.

In [8]:
# Clean stop_id and parent_station values before checking again

stops["stop_id_clean"] = (
    stops["stop_id"]
    .astype(str)
    .str.replace(r"\.0$", "", regex=True)
    .str.strip()
)

stops["parent_station_clean"] = (
    stops["parent_station"]
    .astype(str)
    .str.replace(r"\.0$", "", regex=True)
    .str.strip()
)

# Replace string "nan" with real missing values
stops.loc[stops["parent_station"].isna(), "parent_station_clean"] = pd.NA

stop_ids_clean = set(stops["stop_id_clean"])

parent_station_values_clean = (
    stops["parent_station_clean"]
    .dropna()
)

unique_parent_stations_clean = set(parent_station_values_clean)

parent_station_check_clean = pd.DataFrame({
    "metric": [
        "total stop rows",
        "rows with parent_station",
        "unique parent_station values",
        "parent_station values found as stop_id",
        "parent_station values not found as stop_id"
    ],
    "value": [
        len(stops),
        stops["parent_station_clean"].notna().sum(),
        len(unique_parent_stations_clean),
        len(unique_parent_stations_clean & stop_ids_clean),
        len(unique_parent_stations_clean - stop_ids_clean)
    ]
})

parent_station_check_clean

,metric,value
0,total stop rows,89547
1,rows with parent_station,1556
2,unique parent_station values,658
3,parent_station values found as stop_id,658
4,parent_station values not found as stop_id,0


## Comment

After cleaning the ID values, all 658 unique `parent_station` values were found as `stop_id`.

This means the GTFS stop hierarchy is valid.  
The earlier result was only caused by formatting differences.

Finland has 797 station-level rows with `location_type = 1`, and 1,556 stop/platform rows linked to parent stations.

## Prepare GTFS station-level table

The stop hierarchy check showed that Finland has valid station-level rows.

For the station-level MVD comparison, I use only GTFS rows with `location_type = 1`.  
These rows represent station-level places and are more comparable to NeTEx `StopPlace` than the raw platform/stop rows.

In [9]:
# Prepare GTFS station-level table

gtfs_stations = stops[stops["location_type"] == 1].copy()

gtfs_stations = gtfs_stations[[
    "stop_id",
    "stop_name",
    "stop_lat",
    "stop_lon",
    "location_type"
]].copy()

gtfs_stations.head()

,stop_id,stop_name,stop_lat,stop_lon,location_type
0,300000,Kamppi (lähiliikenneterminaali),60.169008,24.931662,1
1,300001,Elielinaukio,60.171808,24.939488,1
2,300002,Rautatientori,60.171283,24.942572,1
3,300003,Pasila,60.198118,24.934074,1
4,300004,Malmin asema (terminaali),60.250292,25.009293,1


In [10]:
# Quality check for GTFS station-level table

gtfs_station_quality = pd.DataFrame({
    "metric": [
        "GTFS station rows",
        "unique station stop_ids",
        "missing station names",
        "missing latitudes",
        "missing longitudes",
        "duplicate station stop_ids"
    ],
    "value": [
        len(gtfs_stations),
        gtfs_stations["stop_id"].nunique(),
        gtfs_stations["stop_name"].isna().sum(),
        gtfs_stations["stop_lat"].isna().sum(),
        gtfs_stations["stop_lon"].isna().sum(),
        gtfs_stations["stop_id"].duplicated().sum()
    ]
})

gtfs_station_quality

,metric,value
0,GTFS station rows,797
1,unique station stop_ids,797
2,missing station names,0
3,missing latitudes,0
4,missing longitudes,0
5,duplicate station stop_ids,0


## Comment

The GTFS station-level table contains 797 station rows.

The quality check looks clean:

- all station IDs are unique
- no station names are missing
- no coordinates are missing
- there are no duplicate station IDs


## Inspect GTFS route labels

For the line-level comparison, GTFS usually uses `route_short_name`.

Before comparing it later with NeTEx `public_code`, I check whether `route_short_name` is complete and whether several route rows share the same public label.

The labels are kept as strings because some route labels contain letters or may contain leading zeros.

In [11]:
# Clean GTFS route_short_name as string

routes["route_short_name_clean"] = (
    routes["route_short_name"]
    .astype("string")
    .str.strip()
)

# Replace empty strings with missing values
routes.loc[routes["route_short_name_clean"] == "", "route_short_name_clean"] = pd.NA

gtfs_route_label_quality = pd.DataFrame({
    "metric": [
        "GTFS route rows",
        "rows with route_short_name",
        "rows missing route_short_name",
        "unique route_short_name values",
        "duplicate public-label rows"
    ],
    "value": [
        len(routes),
        routes["route_short_name_clean"].notna().sum(),
        routes["route_short_name_clean"].isna().sum(),
        routes["route_short_name_clean"].nunique(),
        routes["route_short_name_clean"].notna().sum() - routes["route_short_name_clean"].nunique()
    ]
})

gtfs_route_label_quality

,metric,value
0,GTFS route rows,7226
1,rows with route_short_name,5450
2,rows missing route_short_name,1776
3,unique route_short_name values,1873
4,duplicate public-label rows,3577


In [12]:
# Show examples of duplicated public route labels

duplicate_route_labels = (
    routes[routes["route_short_name_clean"].notna()]
    .groupby("route_short_name_clean")
    .size()
    .reset_index(name="n_route_rows")
    .sort_values("n_route_rows", ascending=False)
)

duplicate_route_labels.head(10)

,route_short_name_clean,n_route_rows
1283,FlixBus,874
368,3,71
0,1,67
226,2,59
1282,Finferries,47
480,4,46
608,5,40
1,10,31
937,8,31
27,11,30


## Comment

The GTFS route table contains 7,226 route rows.

`route_short_name` is available for 5,450 rows, but it is missing for 1,776 rows.  
There are 1,873 unique public route labels.

Many public labels appear in several technical route rows.  
For example, `FlixBus` appears 874 times, and simple labels such as `1`, `2`, and `3` also appear many times.

As in Norway and Sweden, the line-level comparison should deduplicate routes by public label first.

## Inspect routes without route_short_name

Some Finnish GTFS route rows do not have `route_short_name`.

Before preparing the final GTFS line-level table, I inspect these missing rows.  
This helps decide whether `route_long_name` should be used as a fallback, like in Sweden.

In [13]:
# Inspect rows missing route_short_name

routes_missing_short_name = routes[routes["route_short_name_clean"].isna()].copy()

routes_missing_short_name[[
    "route_id",
    "agency_id",
    "route_short_name",
    "route_long_name",
    "route_type"
]].head(20)

,route_id,agency_id,route_short_name,route_long_name,route_type
1528,23235,1083,NaN,Saarijärvi - Kannonkoski,3
1529,23236,1083,NaN,Kannonkoski - Saarijärvi,3
1553,23512,1162,NaN,Uurainen - Kintaus - Jyväskylä,3
1556,23515,1162,NaN,Iisalmi - Lapinlahti - Kuopio,3
1557,23518,1162,NaN,Nurmes - Nilsiä - Kuopio,3
1558,23519,1162,NaN,Kuopio - Nilsiä - Nurmes,3
1559,23522,1162,NaN,Sallatunturi - Salla - Kemijärvi,3
1560,23523,1162,NaN,Kemijärvi - Salla,3
1565,23573,1049,NaN,Loimaa - Mellilä - Niinijoki - Loimaa,3
1566,23576,1049,NaN,Oripää - Keihäskoski - Nummioja - Yläne,3


In [14]:
# Count missing route_short_name by route_type

missing_short_name_by_type = (
    routes_missing_short_name
    .groupby("route_type")
    .size()
    .reset_index(name="missing_route_short_name_rows")
    .sort_values("missing_route_short_name_rows", ascending=False)
)

missing_short_name_by_type

,route_type,missing_route_short_name_rows
0,3,1729
1,4,47


In [15]:
# Check whether route_long_name is available when route_short_name is missing

missing_short_name_long_name_check = pd.DataFrame({
    "metric": [
        "rows missing route_short_name",
        "of these, rows with route_long_name",
        "of these, rows missing route_long_name",
        "unique route_long_name values among missing route_short_name rows"
    ],
    "value": [
        len(routes_missing_short_name),
        routes_missing_short_name["route_long_name"].notna().sum(),
        routes_missing_short_name["route_long_name"].isna().sum(),
        routes_missing_short_name["route_long_name"].nunique()
    ]
})

missing_short_name_long_name_check

,metric,value
0,rows missing route_short_name,1776
1,"of these, rows with route_long_name",1776
2,"of these, rows missing route_long_name",0
3,unique route_long_name values among missing ro...,1639


## Comment

The missing `route_short_name` values are not empty route records.

All 1,776 rows without `route_short_name` still have a `route_long_name`.  
Most of these rows are bus routes with `route_type = 3`, and a smaller number are ferry routes with `route_type = 4`.

This means `route_long_name` can be used as a fallback label when `route_short_name` is missing.

For the Finnish GTFS line-level table, the best public label is therefore:

`route_short_name` if available, otherwise `route_long_name`.

## Prepare GTFS line-level table

Some Finnish GTFS routes do not have `route_short_name`, but all of them have `route_long_name`.

For the line-level comparison, I therefore create one public GTFS line label:

- use `route_short_name` when available
- otherwise use `route_long_name`

This keeps more routes in the comparison and avoids dropping valid route records.

In [16]:
# Create final GTFS public line label

routes["gtfs_line_label"] = routes["route_short_name_clean"]

routes.loc[
    routes["gtfs_line_label"].isna(),
    "gtfs_line_label"
] = (
    routes.loc[routes["gtfs_line_label"].isna(), "route_long_name"]
    .astype("string")
    .str.strip()
)

# Replace empty strings with missing values
routes.loc[routes["gtfs_line_label"] == "", "gtfs_line_label"] = pd.NA

# Prepare line-level table with one row per route row first
gtfs_lines_raw = routes[[
    "route_id",
    "agency_id",
    "gtfs_line_label",
    "route_short_name",
    "route_long_name",
    "route_type"
]].copy()

gtfs_lines_raw.head()

,route_id,agency_id,gtfs_line_label,route_short_name,route_long_name,route_type
0,20000,1000,1,1,Eira - Kamppi (M) - Töölö - Sörnäinen (M) - Kä...,0
1,20001,1000,10,10,Ullanlinna - Lasipalatsi - Ruskeasuo - Pikku H...,0
2,20002,1000,105,105,Lauttasaari(M)-Westend-Haukilahti-Mankkaa,701
3,20004,1000,10H,10H,Pikku Huopalahti - Ruskeasuo - Ooppera,0
4,20005,1000,111,111,Otaniemi-Tapiola (M)-Westend-Matinkylä (M),701


In [17]:
# Quality check for GTFS line labels

gtfs_line_label_quality = pd.DataFrame({
    "metric": [
        "GTFS route rows",
        "rows with final GTFS line label",
        "rows missing final GTFS line label",
        "unique final GTFS line labels",
        "duplicate public-label rows"
    ],
    "value": [
        len(gtfs_lines_raw),
        gtfs_lines_raw["gtfs_line_label"].notna().sum(),
        gtfs_lines_raw["gtfs_line_label"].isna().sum(),
        gtfs_lines_raw["gtfs_line_label"].nunique(),
        gtfs_lines_raw["gtfs_line_label"].notna().sum() - gtfs_lines_raw["gtfs_line_label"].nunique()
    ]
})

gtfs_line_label_quality

,metric,value
0,GTFS route rows,7226
1,rows with final GTFS line label,7226
2,rows missing final GTFS line label,0
3,unique final GTFS line labels,3512
4,duplicate public-label rows,3714


## Comment

After using `route_long_name` as a fallback, every GTFS route row has a public line label.

The final GTFS line table has:

- 7,226 route rows
- 0 missing line labels
- 3,512 unique public line labels
- 3,714 duplicate public-label rows


For the line-level comparison, the GTFS side should therefore be deduplicated to one row per public line label.

## Deduplicate GTFS lines by public label

The GTFS route table contains many route rows with the same public label.

For the line-level comparison, I do not compare raw route rows.  
Instead, I prepare one GTFS line-level table with one row per public line label.

The transport mode is kept as context.

In [18]:
# Deduplicate GTFS lines by public label

gtfs_lines = (
    gtfs_lines_raw[gtfs_lines_raw["gtfs_line_label"].notna()]
    .groupby("gtfs_line_label")
    .agg(
        n_route_rows=("route_id", "count"),
        route_types=("route_type", lambda x: sorted(x.dropna().unique().tolist())),
        example_route_id=("route_id", "first"),
        example_route_long_name=("route_long_name", "first")
    )
    .reset_index()
)

gtfs_lines.head(10)

,gtfs_line_label,n_route_rows,route_types,example_route_id,example_route_long_name
0,1,67,"[0, 3]",20000,Eira - Kamppi (M) - Töölö - Sörnäinen (M) - Kä...
1,10,31,"[0, 3, 704]",20001,Ullanlinna - Lasipalatsi - Ruskeasuo - Pikku H...
2,100,7,[3],20880,Keskusta-Turkuhalli
3,1001,1,[3],22097,P1 Kiiskinmäki
4,1003,1,[3],22098,P3 Kivisalmi
5,101,6,[3],20881,Keskusta-Logomo
6,1011,1,[3],22100,SaiPa Yliopisto-Keskusta-Jäähalli
7,1012,1,[3],22101,SaiPa Kivisalmi-Keskusta-Jäähalli
8,102,2,[3],21518,Pori-Eurajoki-Rauma
9,102N,1,[701],132522,Kamppi-Lauttasaari(M)-Westend-Tapiola(M)-Teekk...


In [19]:
# Quality check for deduplicated GTFS line table

gtfs_lines_quality = pd.DataFrame({
    "metric": [
        "unique GTFS line labels",
        "missing GTFS line labels",
        "duplicate GTFS line labels after deduplication",
        "line labels with more than one route row"
    ],
    "value": [
        len(gtfs_lines),
        gtfs_lines["gtfs_line_label"].isna().sum(),
        gtfs_lines["gtfs_line_label"].duplicated().sum(),
        (gtfs_lines["n_route_rows"] > 1).sum()
    ]
})

gtfs_lines_quality

,metric,value
0,unique GTFS line labels,3512
1,missing GTFS line labels,0
2,duplicate GTFS line labels after deduplication,0
3,line labels with more than one route row,929


## Comment

The GTFS line table now has one row for each public line label.

There are 3,512 unique GTFS line labels.  
No line labels are missing.

In the original `routes.txt`, the same label could appear more than once.  
For example, line `1` could appear many times and for this comparison, I only need to know that line `1`exists.
I don't need to count every repeated row.

Therefore, I keep each public line label only once.

## Inspect GTFS transport modes


In [20]:
# GTFS route_type meaning
gtfs_route_type_names = {
    # Basic GTFS codes
    0:    "Tram",
    1:    "Metro / Subway",
    2:    "Rail",
    3:    "Bus",
    4:    "Ferry",
    5:    "Cable tram",
    6:    "Aerial lift",
    7:    "Funicular",
    11:   "Trolleybus",
    12:   "Monorail",
    # Extended codes found in Finland
    102:  "Long-distance rail",
    109:  "Suburban rail",
    700:  "Bus",
    701:  "Regional bus",
    702:  "Express bus",
    704:  "Local bus",
    712:  "School bus",
    715:  "Demand-responsive bus",
    900:  "Tram",
    1002: "Ferry",
    1004: "Ferry",
    1008: "Ferry / water transport",
    1104: "Share taxi"
}

gtfs_transport_modes = (
    routes["route_type"]
    .value_counts(dropna=False)
    .reset_index()
)
gtfs_transport_modes.columns = ["route_type", "n_route_rows"]
gtfs_transport_modes["mode_name"] = gtfs_transport_modes["route_type"].map(gtfs_route_type_names)
gtfs_transport_modes

,route_type,n_route_rows,mode_name
0,3,5706,Bus
1,102,596,Long-distance rail
2,701,359,Regional bus
3,1104,224,Share taxi
4,109,89,Suburban rail
5,4,65,Ferry
6,704,60,Local bus
7,1002,47,Ferry
8,0,29,Tram
9,715,18,Demand-responsive bus


In [21]:
# Transport modes in the deduplicated GTFS line table

gtfs_line_modes = (
    gtfs_lines_raw[gtfs_lines_raw["gtfs_line_label"].notna()]
    .groupby("route_type")["gtfs_line_label"]
    .nunique()
    .reset_index(name="n_unique_line_labels")
    .sort_values("n_unique_line_labels", ascending=False)
)

gtfs_line_modes["mode_name"] = gtfs_line_modes["route_type"].map(gtfs_route_type_names)

gtfs_line_modes

,route_type,n_unique_line_labels,mode_name
2,3,2649,Bus
4,102,373,Long-distance rail
6,701,359,Regional bus
14,1104,112,Share taxi
8,704,60,Local bus
3,4,46,Ferry
0,0,27,Tram
5,109,18,Suburban rail
10,715,16,Demand-responsive bus
9,712,14,School bus


## Comment

The Finnish GTFS feed contains several transport modes.

Most route rows are bus routes.  
There are also rail, ferry, tram, metro, share taxi, and other bus-related categories.

Finland uses detailed `route_type` codes, not only the basic GTFS codes.  
That is why I needed the expanded mode mapping.

In the deduplicated line table, bus is still the largest group, with 2,649 unique line labels.


## Inspect GTFS service_id coverage

For the calendar comparison, I first check the GTFS service IDs.

The `trips` table uses `service_id` to connect trips with calendar information.  
A service ID can appear in `calendar.txt`, in `calendar_dates.txt`, or in both.

This step checks whether all services used by trips have calendar information.

In [22]:
# Check GTFS service_id coverage across trips, calendar, and calendar_dates

trip_service_ids = set(trips["service_id"].astype(str))
calendar_service_ids = set(calendar["service_id"].astype(str))
calendar_dates_service_ids = set(calendar_dates["service_id"].astype(str))

all_calendar_service_ids = calendar_service_ids | calendar_dates_service_ids

gtfs_service_id_coverage = pd.DataFrame({
    "metric": [
        "service_ids used in trips",
        "service_ids in calendar.txt",
        "service_ids in calendar_dates.txt",
        "service_ids in calendar or calendar_dates",
        "trip service_ids found in calendar or calendar_dates",
        "trip service_ids missing from both calendar files",
        "service_ids only in calendar_dates.txt"
    ],
    "value": [
        len(trip_service_ids),
        len(calendar_service_ids),
        len(calendar_dates_service_ids),
        len(all_calendar_service_ids),
        len(trip_service_ids & all_calendar_service_ids),
        len(trip_service_ids - all_calendar_service_ids),
        len(calendar_dates_service_ids - calendar_service_ids)
    ]
})

gtfs_service_id_coverage

,metric,value
0,service_ids used in trips,38597
1,service_ids in calendar.txt,25226
2,service_ids in calendar_dates.txt,25858
3,service_ids in calendar or calendar_dates,41974
4,trip service_ids found in calendar or calendar...,38597
5,trip service_ids missing from both calendar files,0
6,service_ids only in calendar_dates.txt,16748


## Comment

Every trip has calendar information.

All 38,597 service IDs used in `trips.txt` are found either in `calendar.txt` or in `calendar_dates.txt`.

But many services exist only as specific dates.  
There are 16,748 service IDs that appear only in `calendar_dates.txt`.

So for Finland, I cannot use only `calendar.txt`.  
I need to combine `calendar.txt` and `calendar_dates.txt` to rebuild the active service dates.

## Inspect GTFS calendar date ranges and exceptions


In [23]:
# Convert GTFS date columns to datetime

calendar["start_date_dt"] = pd.to_datetime(calendar["start_date"].astype(str), format="%Y%m%d")
calendar["end_date_dt"] = pd.to_datetime(calendar["end_date"].astype(str), format="%Y%m%d")

calendar_dates["date_dt"] = pd.to_datetime(calendar_dates["date"].astype(str), format="%Y%m%d")

gtfs_calendar_range_check = pd.DataFrame({
    "metric": [
        "calendar.txt earliest start_date",
        "calendar.txt latest end_date",
        "calendar_dates.txt earliest date",
        "calendar_dates.txt latest date"
    ],
    "value": [
        calendar["start_date_dt"].min(),
        calendar["end_date_dt"].max(),
        calendar_dates["date_dt"].min(),
        calendar_dates["date_dt"].max()
    ]
})

gtfs_calendar_range_check

,metric,value
0,calendar.txt earliest start_date,2014-07-01
1,calendar.txt latest end_date,2080-12-31
2,calendar_dates.txt earliest date,2015-06-01
3,calendar_dates.txt latest date,2030-06-25


In [24]:
# Inspect exception types in calendar_dates.txt

gtfs_exception_type_counts = (
    calendar_dates["exception_type"]
    .value_counts(dropna=False)
    .reset_index()
)

gtfs_exception_type_counts.columns = ["exception_type", "count"]

gtfs_exception_type_counts

,exception_type,count
0,1,716740
1,2,100870


## Comment

The GTFS calendar covers a very long period.

In `calendar.txt`, the earliest start date is 2014-07-01 and the latest end date is 2080-12-31.  
This means some services have very broad validity periods.

The `calendar_dates.txt` file covers dates from 2015-06-01 to 2030-06-25.

Both exception types are used:

- `1` means a service is added on that date
- `2` means a service is removed on that date


## Inspect GTFS feed information

The GTFS calendar has very long dates, which is strange.

Therefore before rebuilding active service dates, I check `feed_info.txt` because sometimes this file gives the real feed start date and end date.


In [25]:
# Check whether feed_info.txt exists and inspect it

with zipfile.ZipFile(gtfs_zip_path, "r") as z:
    gtfs_file_names = z.namelist()

"feed_info.txt" in gtfs_file_names

True

In [26]:
feed_info = read_gtfs_from_zip(gtfs_zip_path, "feed_info.txt")

print("feed_info shape:", feed_info.shape)
print("feed_info columns:")
print(feed_info.columns.tolist())

feed_info.head()

feed_info shape: (1, 9)
feed_info columns:
['feed_publisher_name', 'feed_publisher_url', 'feed_lang', 'default_lang', 'feed_start_date', 'feed_end_date', 'feed_version', 'feed_contact_email', 'feed_contact_url']


,feed_publisher_name,feed_publisher_url,feed_lang,default_lang,feed_start_date,feed_end_date,feed_version,feed_contact_email,feed_contact_url
0,Fintraffic,https://www.fintraffic.fi/fi,fi,NaN,NaN,NaN,20260512034324,koontipalvelu.support@weaselsoftware.fi,NaN


## Comment

The GTFS feed contains a `feed_info.txt` file.

However, `feed_start_date` and `feed_end_date` are missing.  
So this file does not give a clear validity window for the feed.


## Classify GTFS calendar service IDs

Because some calendar rows go until 2080, I do not rebuild all active dates yet.  
First, I classify the service IDs by where they appear.

This helps to understand which services have regular weekly rules and which services are only defined by specific dates.

In [27]:
# Classify GTFS service IDs by calendar source
# (trip_service_ids, calendar_service_ids, calendar_dates_service_ids were already
# computed in the service_id coverage check above and are reused here)
all_service_ids = sorted(calendar_service_ids | calendar_dates_service_ids)

gtfs_service_sources = pd.DataFrame({
    "service_id": all_service_ids
})

gtfs_service_sources["used_in_trips"]       = gtfs_service_sources["service_id"].isin(trip_service_ids)
gtfs_service_sources["in_calendar"]         = gtfs_service_sources["service_id"].isin(calendar_service_ids)
gtfs_service_sources["in_calendar_dates"]   = gtfs_service_sources["service_id"].isin(calendar_dates_service_ids)

def classify_service_source(row):
    if row["in_calendar"] and row["in_calendar_dates"]:
        return "calendar_and_calendar_dates"
    elif row["in_calendar"]:
        return "calendar_only"
    elif row["in_calendar_dates"]:
        return "calendar_dates_only"
    else:
        return "missing"  # cannot occur since all_service_ids is the union of both sets

gtfs_service_sources["source_type"] = gtfs_service_sources.apply(classify_service_source, axis=1)

gtfs_service_source_summary = (
    gtfs_service_sources
    .groupby(["source_type", "used_in_trips"])
    .size()
    .reset_index(name="n_service_ids")
    .sort_values(["source_type", "used_in_trips"])
)
display(gtfs_service_source_summary)

,source_type,used_in_trips,n_service_ids
0,calendar_and_calendar_dates,False,173
1,calendar_and_calendar_dates,True,8937
2,calendar_dates_only,False,2069
3,calendar_dates_only,True,14679
4,calendar_only,False,1135
5,calendar_only,True,14981


In [28]:
# Show examples of date-only services used in trips

gtfs_service_sources[
    (gtfs_service_sources["source_type"] == "calendar_dates_only") &
    (gtfs_service_sources["used_in_trips"])
].head(10)

,service_id,used_in_trips,in_calendar,in_calendar_dates,source_type
12988,10021_209563.458,True,False,True,calendar_dates_only
12989,10021_214174.458,True,False,True,calendar_dates_only
12990,10021_214178.458,True,False,True,calendar_dates_only
12991,10021_215082.458,True,False,True,calendar_dates_only
12992,10021_215097.458,True,False,True,calendar_dates_only
12993,10021_215098.458,True,False,True,calendar_dates_only
12994,10021_215099.458,True,False,True,calendar_dates_only
12995,10021_215100.458,True,False,True,calendar_dates_only
12996,10021_215102.458,True,False,True,calendar_dates_only
12997,10021_215323.458,True,False,True,calendar_dates_only


## Comment

The GTFS calendar uses different types of service definitions.

Some service IDs are in both `calendar.txt` and `calendar_dates.txt`.  
These services have weekly rules and also date exceptions.

Some service IDs are only in `calendar.txt`.  
These services are defined only by weekly rules.

Many service IDs are only in `calendar_dates.txt`.  
These services are defined only by specific dates.

This means the active-date reconstruction must handle all three cases.  
I cannot use only `calendar.txt`, and I also cannot use only `calendar_dates.txt`.

## Inspect long GTFS calendar validity periods


In [29]:
# Inspect most common calendar end dates

calendar_end_date_counts = (
    calendar["end_date_dt"]
    .value_counts()
    .reset_index()
)

calendar_end_date_counts.columns = ["end_date", "n_calendar_rows"]

calendar_end_date_counts = calendar_end_date_counts.sort_values(
    "n_calendar_rows",
    ascending=False
)

calendar_end_date_counts.head(15)

,end_date,n_calendar_rows
0,2026-05-31,4106
1,2026-06-14,3330
2,2026-07-06,2781
3,2026-08-09,1157
4,2026-12-12,1085
5,2026-05-10,703
6,2026-06-21,682
7,2026-06-06,396
8,2026-05-30,381
9,2026-06-05,358


In [30]:
# Count calendar rows with very long validity

long_calendar_validity_check = pd.DataFrame({
    "metric": [
        "calendar rows",
        "rows ending in 2080",
        "rows ending after 2030",
        "rows used in trips ending after 2030"
    ],
    "value": [
        len(calendar),
        (calendar["end_date_dt"].dt.year == 2080).sum(),
        (calendar["end_date_dt"].dt.year > 2030).sum(),
        calendar[
            (calendar["service_id"].astype(str).isin(trip_service_ids)) &
            (calendar["end_date_dt"].dt.year > 2030)
        ].shape[0]
    ]
})

long_calendar_validity_check

,metric,value
0,calendar rows,25226
1,rows ending in 2080,2
2,rows ending after 2030,11
3,rows used in trips ending after 2030,7


## Comment

Most GTFS calendar rows end in 2026.

Only 2 calendar rows end in 2080.  
Only 11 rows end after 2030.

So the very long dates are not common in the Finnish GTFS calendar.

This means the 2080 date is only a small special case, not the normal structure of the feed.


## Rebuild GTFS active service dates

GTFS service validity is split across two files:

- `calendar.txt` gives the regular weekly pattern
- `calendar_dates.txt` adds or removes specific dates

To know when each `service_id` is active, I combine both files.

In [31]:
weekday_cols = [
    "monday", "tuesday", "wednesday", "thursday",
    "friday", "saturday", "sunday"
]

comparison_start = calendar_dates["date_dt"].min()
comparison_end   = calendar_dates["date_dt"].max()

print("GTFS active-date window:")
print("Start:", comparison_start)
print("End:  ", comparison_end)

GTFS active-date window:
Start: 2015-06-01 00:00:00
End:   2030-06-25 00:00:00


In [32]:
# Prepare calendar_dates additions and removals
calendar_dates_work = calendar_dates.copy()
calendar_dates_work["service_id"] = calendar_dates_work["service_id"].astype(str)

added_dates   = defaultdict(set)
removed_dates = defaultdict(set)

total = len(calendar_dates_work)
for i, (_, row) in enumerate(calendar_dates_work.iterrows(), start=1):
    print(f"  [{i}/{total}]", end="\r")
    sid  = row["service_id"]
    date = row["date_dt"]
    if row["exception_type"] == 1:
        added_dates[sid].add(date)
    elif row["exception_type"] == 2:
        removed_dates[sid].add(date)

print(f"\nDone — {len(added_dates)} service IDs with added dates, "
      f"{len(removed_dates)} with removed dates.")

  [817610/817610]
Done — 21661 service IDs with added dates, 5249 with removed dates.


In [33]:
# Rebuild active dates from calendar.txt plus calendar_dates.txt
calendar_work = calendar.copy()
calendar_work["service_id"] = calendar_work["service_id"].astype(str)

gtfs_active_date_rows = []
total = len(calendar_work)

for i, (_, row) in enumerate(calendar_work.iterrows(), start=1):
    print(f"  calendar.txt [{i}/{total}]", end="\r")
    sid   = row["service_id"]
    start = max(row["start_date_dt"], comparison_start)
    end   = min(row["end_date_dt"],   comparison_end)
    if start <= end:
        all_days     = pd.date_range(start, end, freq="D")
        active_dates = set()
        for d in all_days:
            if row[weekday_cols[d.weekday()]] == 1:
                active_dates.add(d)
        # Apply exceptions, filtering added dates to comparison window
        active_dates = (active_dates - removed_dates[sid]) | {
            d for d in added_dates[sid]
            if comparison_start <= d <= comparison_end
        }
        for d in sorted(active_dates):
            gtfs_active_date_rows.append({
                "service_id":  sid,
                "active_date": d
            })

print(f"\nDone — calendar.txt processed.")

# Add services that exist only in calendar_dates.txt
calendar_only_ids = set(calendar_work["service_id"])
date_only_ids     = set(calendar_dates_work["service_id"]) - calendar_only_ids

for i, sid in enumerate(date_only_ids, start=1):
    print(f"  calendar_dates_only [{i}/{len(date_only_ids)}]", end="\r")
    for d in sorted(added_dates[sid]):
        if comparison_start <= d <= comparison_end:
            gtfs_active_date_rows.append({
                "service_id":  sid,
                "active_date": d
            })

print(f"\nDone — {len(gtfs_active_date_rows)} total active date rows built.")

gtfs_active_dates = pd.DataFrame(gtfs_active_date_rows)
display(gtfs_active_dates.head())

  calendar.txt [25226/25226]
Done — calendar.txt processed.
  calendar_dates_only [16748/16748]
Done — 1139498 total active date rows built.


,service_id,active_date
0,10040_Livi_KOULPV_M_P_Koski_Tl,2015-06-01
1,10040_Livi_KOULPV_M_P_Koski_Tl,2015-06-02
2,10040_Livi_KOULPV_M_P_Koski_Tl,2015-06-03
3,10040_Livi_KOULPV_M_P_Koski_Tl,2015-06-04
4,10040_Livi_KOULPV_M_P_Koski_Tl,2015-06-05


In [34]:
# Quality check
gtfs_active_dates_quality = pd.DataFrame({
    "metric": [
        "active-date rows",
        "unique service_ids with active dates",
        "earliest active date",
        "latest active date"
    ],
    "value": [
        len(gtfs_active_dates),
        gtfs_active_dates["service_id"].nunique(),
        gtfs_active_dates["active_date"].min(),
        gtfs_active_dates["active_date"].max()
    ]
})
display(gtfs_active_dates_quality)

,metric,value
0,active-date rows,1139498
1,unique service_ids with active dates,40991
2,earliest active date,2015-06-01 00:00:00
3,latest active date,2030-06-25 00:00:00


## Comment

The active-date window goes from 2015-06-01 to 2030-06-25.  
This window comes from `calendar_dates.txt`.

The final table has 1,139,498 active-date rows.  
There are 40,991 service IDs with at least one active date.

This means the GTFS calendar reconstruction worked.

The next check is to see whether all service IDs used in `trips.txt` also have at least one active date in the rebuilt table.

## Check active-date coverage for trip service IDs

After rebuilding the GTFS active dates, I check whether all service IDs used in `trips.txt` have at least one active date.

This is important because the calendar comparison should be based on services that are actually used by trips.

In [35]:
# Check whether trip service IDs have active dates

active_service_ids = set(gtfs_active_dates["service_id"].astype(str))

gtfs_trip_active_date_coverage = pd.DataFrame({
    "metric": [
        "service_ids used in trips",
        "trip service_ids with active dates",
        "trip service_ids without active dates"
    ],
    "value": [
        len(trip_service_ids),
        len(trip_service_ids & active_service_ids),
        len(trip_service_ids - active_service_ids)
    ]
})

gtfs_trip_active_date_coverage

,metric,value
0,service_ids used in trips,38597
1,trip service_ids with active dates,38310
2,trip service_ids without active dates,287


## Comment

Most trip service IDs have active dates.

Out of 38,597 service IDs used in `trips.txt`, 38,310 have at least one active date.

287 service IDs used in trips do not have active dates in the rebuilt table.

This is a small number, but it is worth checking it.

## Inspect trip service IDs without active dates


In [36]:
# Find trip service IDs that have no active dates
# (trip_service_ids and active_service_ids were already computed above and are reused here)
trip_service_ids_without_active_dates = trip_service_ids - active_service_ids

print(f"Trip service IDs with no active dates: {len(trip_service_ids_without_active_dates)}")

Trip service IDs with no active dates: 287


In [37]:
# Check where these missing service IDs are defined

missing_active_service_check = gtfs_service_sources[
    gtfs_service_sources["service_id"].isin(trip_service_ids_without_active_dates)
].copy()

missing_active_service_check["source_type"].value_counts(dropna=False)

source_type
calendar_and_calendar_dates    287
Name: count, dtype: int64

In [38]:
# Show examples

missing_active_service_check.head(20)

,service_id,used_in_trips,in_calendar,in_calendar_dates,source_type
54,10000_1001H5_20260601_20260614_Ti,True,True,True,calendar_and_calendar_dates
84,10000_1001H7_20260601_20260614_Ti,True,True,True,calendar_and_calendar_dates
106,10000_1001T4_20260601_20260614_Ti,True,True,True,calendar_and_calendar_dates
125,10000_1001T5_20260601_20260614_Ti,True,True,True,calendar_and_calendar_dates
153,10000_1001T_20260601_20260614_Ti,True,True,True,calendar_and_calendar_dates
214,10000_1002 4_20260601_20260614_Ti,True,True,True,calendar_and_calendar_dates
264,10000_1002H4_20260601_20260614_Ti,True,True,True,calendar_and_calendar_dates
294,10000_1002H6_20260601_20260614_Ti,True,True,True,calendar_and_calendar_dates
328,10000_1002_20260601_20260614_Ti,True,True,True,calendar_and_calendar_dates
362,10000_1003 4_20260601_20260614_Ti,True,True,True,calendar_and_calendar_dates


## Explore an example

In [39]:
# Pick a deterministic example (sorted, so it is stable across reruns)
example_missing_sid = (
    missing_active_service_check["service_id"]
    .sort_values()
    .iloc[0]
)

example_missing_sid

'10000_1001H5_20260601_20260614_Ti'

In [40]:
# Inspect this service in calendar.txt
calendar[
    calendar["service_id"].astype(str) == example_missing_sid
]

,service_id,monday,tuesday,wednesday,thursday,friday,saturday,sunday,start_date,end_date,start_date_dt,end_date_dt
11016,10000_1001H5_20260601_20260614_Ti,0,1,0,0,0,0,0,20260601,20260614,2026-06-01,2026-06-14


In [41]:
# Inspect this service in calendar_dates.txt
calendar_dates[
    calendar_dates["service_id"].astype(str) == example_missing_sid
].sort_values("date_dt")

,service_id,date,exception_type,date_dt
347977,10000_1001H5_20260601_20260614_Ti,20260602,2,2026-06-02
372645,10000_1001H5_20260601_20260614_Ti,20260609,2,2026-06-09


## Comment

This example explains why some trip service IDs have no active dates.

The service is active only on Tuesdays between 2026-06-01 and 2026-06-14.

There are two Tuesdays in this period: 2026-06-02 and 2026-06-09.  
Both dates are removed in `calendar_dates.txt` with `exception_type = 2`.

So after applying the exceptions, this service has no active dates left.

This means the result is caused by the GTFS data logic, not by an error in the reconstruction code.

## Prepare GTFS calendar summary

The active dates were rebuilt from `calendar.txt` and `calendar_dates.txt`.

Now I create one summary row per `service_id`.  
This gives a simpler calendar-level table for the later comparison with NeTEx.

For each service ID, I keep:

- number of active days
- first active date
- last active date

In [42]:
# Prepare GTFS calendar summary

gtfs_calendar_summary = (
    gtfs_active_dates
    .groupby("service_id")
    .agg(
        n_active_days=("active_date", "count"),
        first_active_date=("active_date", "min"),
        last_active_date=("active_date", "max")
    )
    .reset_index()
)

gtfs_calendar_summary.head()

,service_id,n_active_days,first_active_date,last_active_date
0,10000_1001 5_20260508_20260510_La,1,2026-05-09,2026-05-09
1,10000_1001 5_20260508_20260510_Pe,1,2026-05-08,2026-05-08
2,10000_1001 5_20260508_20260510_Su,1,2026-05-10,2026-05-10
3,10000_1001 5_20260511_20260531_1R,1,2026-05-31,2026-05-31
4,10000_1001 5_20260511_20260531_HR,1,2026-05-16,2026-05-16


In [43]:
gtfs_calendar_summary_quality = pd.DataFrame({
    "metric": [
        "GTFS service IDs with active dates",
        "minimum active days",
        "maximum active days",
        "earliest first active date",
        "latest last active date"
    ],
    "value": [
        len(gtfs_calendar_summary),
        gtfs_calendar_summary["n_active_days"].min(),
        gtfs_calendar_summary["n_active_days"].max(),
        gtfs_calendar_summary["first_active_date"].min(),
        gtfs_calendar_summary["last_active_date"].max()
    ]
})

gtfs_calendar_summary_quality

,metric,value
0,GTFS service IDs with active dates,40991
1,minimum active days,1
2,maximum active days,5504
3,earliest first active date,2015-06-01 00:00:00
4,latest last active date,2030-06-25 00:00:00


## Comment

The GTFS calendar summary is ready.

There are 40,991 service IDs with at least one active date.

Some services are active for only one day.  
The longest service is active for 5,504 days, which is possible because the chosen window goes from 2015 to 2030.

The calendar summary now gives one row per service ID.  
For each service ID, it shows how many days it is active and its first and last active date.

This table will be used later for the GTFS–Netex calendar comparison.

# Part 2: NeTEx Exploration

In [44]:
# Set NeTEx path
netex_zip_path = data_dir / "finland_netex.zip"
print("NeTEx path:")
print(netex_zip_path)
print("\nNeTEx file exists:")
print(netex_zip_path.exists())

NeTEx path:
data/finland_netex.zip

NeTEx file exists:
True


In [45]:
# Inspect NeTEx ZIP contents
with zipfile.ZipFile(netex_zip_path, "r") as z:
    netex_files = pd.DataFrame([
        {
            "name": info.filename,
            "size_mb": round(info.file_size / (1024 * 1024), 2)
        }
        for info in z.infolist()
    ])

display(netex_files.sort_values("size_mb", ascending=False).head(30))

print("Number of files in NeTEx ZIP:", len(netex_files))

,name,size_mb
1,_shared_data.xml,1179.73
0,_stops.xml,69.81
1087,line_20135.xml,57.52
1158,line_20225.xml,54.34
221,line_115268.xml,53.51
189,line_115236.xml,51.02
1044,line_20080.xml,50.49
1204,line_20273.xml,50.47
1136,line_20203.xml,50.06
1157,line_20224.xml,49.29


Number of files in NeTEx ZIP: 7013


## Comment

Finland’s NeTEx is not one small file.
It is a big structured export:
    
_stops.xml        → stops / StopPlaces

_shared_data.xml  → shared calendar and reference data

line_*.xml        → line-level data

In [46]:
# Inspect NeTEx file groups

netex_file_names = netex_files["name"].tolist()

stops_files = [f for f in netex_file_names if f.endswith("_stops.xml") or f == "_stops.xml"]
shared_files = [f for f in netex_file_names if "shared" in f.lower()]
line_files = [f for f in netex_file_names if f.startswith("line_") and f.endswith(".xml")]

netex_file_group_summary = pd.DataFrame({
    "file_group": [
        "stops files",
        "shared data files",
        "line files",
        "all XML files"
    ],
    "count": [
        len(stops_files),
        len(shared_files),
        len(line_files),
        len(netex_file_names)
    ]
})

netex_file_group_summary

,file_group,count
0,stops files,1
1,shared data files,1
2,line files,7011
3,all XML files,7013


## NeTEx structure check: element counter

Before extracting any data from the Finnish NeTEx files, I first check which
elements are present and how many there are. This avoids running slow extractions
on files that may not contain the expected data.

This function reads an XML file inside a ZIP without loading it fully into memory,
counts how many times each target element appears, and returns a summary table.
An optional `max_events` limit allows stopping early for very large files.

In [47]:
def local_name(tag):
    return tag.split("}")[-1] if "}" in tag else tag

def count_target_tags_in_xml(zip_path, xml_file, target_tags, max_events=None):
  
    # Memory-safe XML tag counter
    # If max_events is given, it stops after that many parsed end-events
  
    counts = {tag: 0 for tag in target_tags}
    scanned_events = 0

    with zipfile.ZipFile(zip_path, "r") as z:
        with z.open(xml_file) as f:
            for event, elem in ET.iterparse(f, events=("end",)):
                scanned_events += 1
                tag = local_name(elem.tag)

                if tag in counts:
                    counts[tag] += 1

                elem.clear()

                if max_events is not None and scanned_events >= max_events:
                    break

    rows = []
    for tag, count in counts.items():
        rows.append({
            "file": xml_file,
            "tag": tag,
            "count_in_scan": count,
            "scanned_events": scanned_events,
            "scan_limited": max_events is not None
        })

    return pd.DataFrame(rows)

In [48]:
# Inspect important stop-related tags in _stops.xml

stops_file = "_stops.xml"

netex_stops_tag_check = count_target_tags_in_xml(
    netex_zip_path,
    stops_file,
    target_tags=[
        "StopPlace",
        "Quay",
        "ScheduledStopPoint"
    ],
    max_events=None
)

netex_stops_tag_check

,file,tag,count_in_scan,scanned_events,scan_limited
0,_stops.xml,StopPlace,137227,2519667,False
1,_stops.xml,Quay,138571,2519667,False
2,_stops.xml,ScheduledStopPoint,0,2519667,False


In [49]:
# Inspect important line-related tags in a few line files

sample_line_files = line_files[:5]

line_tag_checks = []

for xml_file in sample_line_files:
    check = count_target_tags_in_xml(
        netex_zip_path,
        xml_file,
        target_tags=[
            "Line",
            "Route",
            "ServiceJourney",
            "PublicCode",
            "TransportMode"
        ],
        max_events=None
    )
    line_tag_checks.append(check)

netex_line_tag_check = pd.concat(line_tag_checks, ignore_index=True)

netex_line_tag_check

,file,tag,count_in_scan,scanned_events,scan_limited
0,line_100935.xml,Line,1,337,False
1,line_100935.xml,Route,1,337,False
2,line_100935.xml,ServiceJourney,2,337,False
3,line_100935.xml,PublicCode,1,337,False
4,line_100935.xml,TransportMode,1,337,False
5,line_101185.xml,Line,1,785,False
6,line_101185.xml,Route,1,785,False
7,line_101185.xml,ServiceJourney,1,785,False
8,line_101185.xml,PublicCode,1,785,False
9,line_101185.xml,TransportMode,1,785,False


In [50]:
# Inspect shared-data tags with a limited scan first

shared_file = "_shared_data.xml"

netex_shared_tag_check_limited = count_target_tags_in_xml(
    netex_zip_path,
    shared_file,
    target_tags=[
        "DayType",
        "DayTypeAssignment",
        "OperatingPeriod",
        "UicOperatingPeriod",
        "ValidDayBits",
        "FromDate",
        "ToDate"
    ],
    max_events=500000
)

netex_shared_tag_check_limited

,file,tag,count_in_scan,scanned_events,scan_limited
0,_shared_data.xml,DayType,0,500000,True
1,_shared_data.xml,DayTypeAssignment,0,500000,True
2,_shared_data.xml,OperatingPeriod,0,500000,True
3,_shared_data.xml,UicOperatingPeriod,0,500000,True
4,_shared_data.xml,ValidDayBits,0,500000,True
5,_shared_data.xml,FromDate,0,500000,True
6,_shared_data.xml,ToDate,0,500000,True


## Comment

The Finnish NeTEx ZIP has a clear structure.

There is one `_stops.xml` file, one `_shared_data.xml` file, and many `line_*.xml` files.

The `_stops.xml` file contains many `StopPlace` and `Quay` elements.  
For the station-level comparison, `StopPlace` is the most relevant element.

The sampled line files contain `Line`, `Route`, `ServiceJourney`, `PublicCode`, and `TransportMode`.  
This means the line files can be used later for the NeTEx line-level table.

The limited scan of `_shared_data.xml` did not find calendar tags yet.  
But the file is very large, so this only means the tags were not found in the first scanned part.

## Inspect sample NeTEx StopPlace records

Before extracting all StopPlaces, I inspect a small sample of 10 records from
`_stops.xml` to confirm that names and coordinates are available and to understand
the ID format used in the Finnish NeTEx feed.


In [51]:
def extract_stopplace_sample(zip_path, xml_file, n_sample=10):
    samples = []
    current = None
    depth   = 0

    with zipfile.ZipFile(zip_path, "r") as z:
        with z.open(xml_file) as f:
            for event, elem in ET.iterparse(f, events=("start", "end")):
                tag = local_name(elem.tag)

                if event == "start" and tag == "StopPlace":
                    depth += 1
                    if depth == 1:
                        current = {
                            "stopplace_id": elem.attrib.get("id"),
                            "name":         None,
                            "latitude":     None,
                            "longitude":    None
                        }

                elif event == "end" and tag == "StopPlace":
                    if depth == 1 and current is not None:
                        samples.append(current)
                        current = None
                    depth -= 1
                    elem.clear()
                    if len(samples) >= n_sample:
                        break

                elif event == "end" and current is not None and depth > 0:
                    text = elem.text.strip() if elem.text else None
                    if tag == "Name" and current["name"] is None:
                        current["name"] = text
                    elif tag == "Latitude" and current["latitude"] is None:
                        current["latitude"] = text
                    elif tag == "Longitude" and current["longitude"] is None:
                        current["longitude"] = text
                    elem.clear()

                elif event == "end":
                    elem.clear()

    return pd.DataFrame(samples)

netex_stopplace_sample = extract_stopplace_sample(
    netex_zip_path,
    "_stops.xml",
    n_sample=10
)
display(netex_stopplace_sample)

,stopplace_id,name,latitude,longitude
0,FSR:StopPlace:425214,Aiamaa,57.7634395,26.0888455
1,FSR:StopPlace:425215,Käärikmäe,57.9547695,25.9695268
2,FSR:StopPlace:425216,Ristimäe,57.9409961,25.9896956
3,FSR:StopPlace:425217,Kullimäe,57.9181182,26.0252685
4,FSR:StopPlace:425218,Kullimäe,57.9180489,26.0261587
5,FSR:StopPlace:425219,Möldre,58.027284,25.8767663
6,FSR:StopPlace:425210,Pagari,57.768799,26.0700659
7,FSR:StopPlace:425211,Laatsi,57.7671681,26.0768638
8,FSR:StopPlace:425212,Laatsi,57.7672195,26.0776207
9,FSR:StopPlace:425213,Aiamaa,57.7628643,26.0902154


## Comment

The StopPlace sample was extracted successfully.

Each row has a StopPlace ID, name, latitude, and longitude.  
So `_stops.xml` can be used for the NeTEx stop-level extraction.


## Check StopPlace coordinate coverage

Before extracting all StopPlaces, I check the coordinate range across a larger
sample of 100 records. This is to verify that the Finnish NeTEx feed covers the
correct geographic area, since the first 10 sampled stops had coordinates near
the Estonian border which looked unexpected at first glance.

In [52]:
netex_stopplace_sample = extract_stopplace_sample(
    netex_zip_path,
    "_stops.xml",
    n_sample=100
)
display(netex_stopplace_sample[["latitude", "longitude"]].astype(float).describe())

,latitude,longitude
count,100.000000,100.000000
mean,59.680998,24.854390
std,1.192079,2.079932
min,57.762864,21.070890
25%,58.071250,22.004101
50%,60.339525,26.064196
75%,60.549165,26.183821
max,61.887636,26.990804


## Comment

The coordinate range is correct. The mean latitude of **59.68°N** and longitude
of **24.85°E** place the sample firmly in the Helsinki region, which is the most
densely served area in Finland.

The minimum latitude of **57.76°N** corresponds to the southernmost stops, likely
near the coast or covering cross-border ferry connections to Estonia. This is
expected for a national feed and is not a data quality issue.

The coordinates are valid and the NeTEx feed covers Finnish territory as expected.

## Extract all NeTEx StopPlaces

The StopPlace sample showed that ID, name, latitude, and longitude can be extracted correctly.

Now I extract all StopPlaces from `_stops.xml`.

This creates the NeTEx stop-level table for the later comparison with GTFS stations.

In [53]:
def extract_all_stopplaces(zip_path, xml_file):
    rows = []
    current = None
    depth = 0

    with zipfile.ZipFile(zip_path, "r") as z:
        with z.open(xml_file) as f:
            for event, elem in ET.iterparse(f, events=("start", "end")):
                tag = local_name(elem.tag)

                if event == "start" and tag == "StopPlace":
                    depth += 1
                    if depth == 1:
                        current = {
                            "stopplace_id": elem.attrib.get("id"),
                            "name":         None,
                            "latitude":     None,
                            "longitude":    None
                        }

                elif event == "end" and tag == "StopPlace":
                    if depth == 1 and current is not None:
                        rows.append(current)
                        if len(rows) % 1000 == 0:
                            print(f"  {len(rows)} StopPlaces extracted...", end="\r")
                        current = None
                    depth -= 1
                    elem.clear()

                elif event == "end" and current is not None and depth > 0:
                    text = elem.text.strip() if elem.text else None
                    if tag == "Name" and current["name"] is None:
                        current["name"] = text
                    elif tag == "Latitude" and current["latitude"] is None:
                        current["latitude"] = text
                    elif tag == "Longitude" and current["longitude"] is None:
                        current["longitude"] = text
                    elem.clear()

                elif event == "end":
                    elem.clear()

    print(f"\nDone — {len(rows)} StopPlaces extracted.")
    return pd.DataFrame(rows)

netex_stopplaces = extract_all_stopplaces(netex_zip_path, "_stops.xml")

netex_stopplaces["latitude"]  = pd.to_numeric(netex_stopplaces["latitude"],  errors="coerce")
netex_stopplaces["longitude"] = pd.to_numeric(netex_stopplaces["longitude"], errors="coerce")

display(netex_stopplaces.head(10))

  137000 StopPlaces extracted...
Done — 137227 StopPlaces extracted.


,stopplace_id,name,latitude,longitude
0,FSR:StopPlace:425214,Aiamaa,57.763439,26.088846
1,FSR:StopPlace:425215,Käärikmäe,57.954769,25.969527
2,FSR:StopPlace:425216,Ristimäe,57.940996,25.989696
3,FSR:StopPlace:425217,Kullimäe,57.918118,26.025268
4,FSR:StopPlace:425218,Kullimäe,57.918049,26.026159
5,FSR:StopPlace:425219,Möldre,58.027284,25.876766
6,FSR:StopPlace:425210,Pagari,57.768799,26.070066
7,FSR:StopPlace:425211,Laatsi,57.767168,26.076864
8,FSR:StopPlace:425212,Laatsi,57.767220,26.077621
9,FSR:StopPlace:425213,Aiamaa,57.762864,26.090215


In [54]:
# Quality check for NeTEx StopPlaces
# (latitude/longitude were already converted to numeric at the end of the extraction cell above)

netex_stopplace_quality = pd.DataFrame({
    "metric": [
        "NeTEx StopPlace rows",
        "unique StopPlace IDs",
        "missing StopPlace IDs",
        "missing names",
        "missing latitudes",
        "missing longitudes",
        "duplicate StopPlace IDs"
    ],
    "value": [
        len(netex_stopplaces),
        netex_stopplaces["stopplace_id"].nunique(),
        netex_stopplaces["stopplace_id"].isna().sum(),
        netex_stopplaces["name"].isna().sum(),
        netex_stopplaces["latitude"].isna().sum(),
        netex_stopplaces["longitude"].isna().sum(),
        netex_stopplaces["stopplace_id"].duplicated().sum()
    ]
})

netex_stopplace_quality

,metric,value
0,NeTEx StopPlace rows,137227
1,unique StopPlace IDs,137227
2,missing StopPlace IDs,0
3,missing names,0
4,missing latitudes,0
5,missing longitudes,0
6,duplicate StopPlace IDs,0


## Comment

The NeTEx StopPlace extraction worked.

There are 137,227 StopPlace rows.  
All StopPlace IDs are unique.

No names or coordinates are missing.  

One important point: NeTEx has many more StopPlaces than the 797 GTFS station rows.  

- GTFS station rows: 797

- NeTEx StopPlace rows: 137,227

In [55]:
print(f"Total GTFS stops: {len(stops)}")

Total GTFS stops: 89547


# Check whether Finnish NeTEx StopPlaces have a parent-child structure

In [56]:
# Check if StopPlaces in Finland have a ParentSiteRef (indicating hierarchy)
result = count_target_tags_in_xml(
    netex_zip_path,
    "_stops.xml",
    target_tags=["ParentSiteRef", "quays", "Quay"],
    max_events=None
)
display(result)

,file,tag,count_in_scan,scanned_events,scan_limited
0,_stops.xml,ParentSiteRef,0,2519667,False
1,_stops.xml,quays,137227,2519667,False
2,_stops.xml,Quay,138571,2519667,False


## Comment

- `ParentSiteRef`: 0 — no StopPlace has a parent reference, so there is no hierarchy. All StopPlaces are flat, top-level entries
    
- `quays`: 137,227 — every single StopPlace has a quays container element
    
- `Quay`: 138,571 — slightly more Quays than StopPlaces, meaning most StopPlaces have one Quay each, with a few having two

StopPlace is the right level to compare against GTFS stops. Each StopPlace represents one stop location with coordinates, which is exactly what a GTFS stop is. There is no hierarchy since ParentSiteRef is 0 everywhere.
So the approach is confirmed: all GTFS stops vs all NeTEx StopPlaces by coordinate proximity

---
> ## Note on stop-level comparison approach for Finland
>
> For Finland, I do not use only GTFS station-level rows with `location_type = 1`.
> The reason is that the GTFS station-level table has only 797 rows, while NeTEx
> has 137,227 StopPlaces. This is too different for a fair station-level comparison.
>
> The full GTFS stops table has 89,547 rows, which is much closer to the NeTEx
> StopPlace count. This suggests that, in Finland, many NeTEx StopPlaces represent
> individual GTFS stops rather than only parent stations.
>
> Therefore, for Finland, the stop-level comparison should use all GTFS stops
> and compare them with NeTEx StopPlaces.
---

## Prepare GTFS stop-level table for Finland

For Finland, the main stop-level comparison uses all GTFS stops, not only station-level rows.
The reason is that NeTEx StopPlaces are much closer in count to all GTFS stops than to GTFS parent stations.

| Variable | Description |
|---|---|
| `gtfs_stations` | Exploration table — only `location_type = 1` |
| `gtfs_stops_all` | Main table for Finland stop comparison |

In [57]:
# Prepare all GTFS stops for Finland stop-level comparison

gtfs_stops_all = stops[[
    "stop_id",
    "stop_name",
    "stop_lat",
    "stop_lon",
    "location_type",
    "parent_station"
]].copy()

gtfs_stops_all["stop_id"] = gtfs_stops_all["stop_id"].astype(str)
gtfs_stops_all["stop_name"] = gtfs_stops_all["stop_name"].astype("string").str.strip()

gtfs_stops_all["stop_lat"] = pd.to_numeric(gtfs_stops_all["stop_lat"], errors="coerce")
gtfs_stops_all["stop_lon"] = pd.to_numeric(gtfs_stops_all["stop_lon"], errors="coerce")

gtfs_stops_all.head()

,stop_id,stop_name,stop_lat,stop_lon,location_type,parent_station
0,300000,Kamppi (lähiliikenneterminaali),60.169008,24.931662,1,NaN
1,300001,Elielinaukio,60.171808,24.939488,1,NaN
2,300002,Rautatientori,60.171283,24.942572,1,NaN
3,300003,Pasila,60.198118,24.934074,1,NaN
4,300004,Malmin asema (terminaali),60.250292,25.009293,1,NaN


In [58]:
gtfs_stops_all_quality = pd.DataFrame({
    "metric": [
        "GTFS stop rows",
        "unique stop IDs",
        "missing stop names",
        "missing latitudes",
        "missing longitudes",
        "duplicate stop IDs"
    ],
    "value": [
        len(gtfs_stops_all),
        gtfs_stops_all["stop_id"].nunique(),
        gtfs_stops_all["stop_name"].isna().sum(),
        gtfs_stops_all["stop_lat"].isna().sum(),
        gtfs_stops_all["stop_lon"].isna().sum(),
        gtfs_stops_all["stop_id"].duplicated().sum()
    ]
})

gtfs_stops_all_quality

,metric,value
0,GTFS stop rows,89547
1,unique stop IDs,89547
2,missing stop names,0
3,missing latitudes,0
4,missing longitudes,0
5,duplicate stop IDs,0


## Inspect sample NeTEx Lines

The next step is to inspect the line files.  
First I extract a small sample of NeTEx lines to check whether the line ID, name, public code, and transport mode can be read correctly.

In [59]:
def extract_line_sample(zip_path, line_files, n_sample=10):
    rows = []
    with zipfile.ZipFile(zip_path, "r") as z:
        for xml_file in line_files[:n_sample]:
            current = {
                "file":           xml_file,
                "line_id":        None,
                "name":           None,
                "public_code":    None,
                "transport_mode": None
            }
            inside_line = False
            with z.open(xml_file) as f:
                for event, elem in ET.iterparse(f, events=("start", "end")):
                    tag = local_name(elem.tag)
                    if event == "start" and tag == "Line":
                        inside_line = True
                        current["line_id"] = elem.attrib.get("id")
                    elif event == "end" and inside_line:
                        text = elem.text.strip() if elem.text else None
                        if tag == "Name" and current["name"] is None:
                            current["name"] = text
                        elif tag == "PublicCode" and current["public_code"] is None:
                            current["public_code"] = text
                        elif tag == "TransportMode" and current["transport_mode"] is None:
                            current["transport_mode"] = text
                        elif tag == "Line":
                            rows.append(current)
                            elem.clear()
                            break
                        elem.clear()
                    elif event == "end":
                        elem.clear()
    return pd.DataFrame(rows)

netex_line_sample = extract_line_sample(
    netex_zip_path,
    line_files,
    n_sample=10
)
display(netex_line_sample)

,file,line_id,name,public_code,transport_mode
0,line_100935.xml,FSR:Line:100935,Mietoinen-Askainen-Lemu,127,bus
1,line_101185.xml,FSR:Line:101185,Vuosnainen - Kustavi - Turku,125,bus
2,line_101186.xml,FSR:Line:101186,Turku - Kustavi - Vuosnainen,125,bus
3,line_101193.xml,FSR:Line:101193,Tolkkinen - Sairaala,1,bus
4,line_101194.xml,FSR:Line:101194,Sairaala - Tolkkinen,1,bus
5,line_101195.xml,FSR:Line:101195,Tolkkinen - Sairaala,1V,bus
6,line_101196.xml,FSR:Line:101196,Sairaala - Tolkkinen,1V,bus
7,line_101197.xml,FSR:Line:101197,Tarmola - Hamari,2,bus
8,line_101198.xml,FSR:Line:101198,Hamari - Tarmola,2,bus
9,line_101199.xml,FSR:Line:101199,Tuomiokirkko - Sairaala,5,bus


## Comment

The sample already shows that multiple NeTEx Line IDs can share the same `PublicCode`. 

For example:

- FSR:Line:101185 → PublicCode 125

- FSR:Line:101186 → PublicCode 125

This means the public line code is likely the better comparison field for the GTFS–NeTEx line-level comparison.

So for the later comparison: **GTFS route_short_name ↔ NeTEx PublicCode**

## Extract all NeTEx lines

After validating the sample extraction, I now extract all NeTEx lines from the line XML files.


In [60]:
def extract_all_netex_lines(zip_path, line_files):
    rows = []
    total = len(line_files)
    with zipfile.ZipFile(zip_path, "r") as z:
        for i, xml_file in enumerate(line_files, start=1):
            if i == 1 or i % 100 == 0 or i == total:
                print(f"  [{i}/{total}] {xml_file}", end="\r")
            current = {
                "file":           xml_file,
                "line_id":        None,
                "name":           None,
                "public_code":    None,
                "transport_mode": None
            }
            inside_line = False
            with z.open(xml_file) as f:
                for event, elem in ET.iterparse(f, events=("start", "end")):
                    tag = local_name(elem.tag)
                    if event == "start" and tag == "Line":
                        inside_line = True
                        current["line_id"] = elem.attrib.get("id")
                    elif event == "end" and inside_line:
                        text = elem.text.strip() if elem.text else None
                        if tag == "Name" and current["name"] is None:
                            current["name"] = text
                        elif tag == "PublicCode" and current["public_code"] is None:
                            current["public_code"] = text
                        elif tag == "TransportMode" and current["transport_mode"] is None:
                            current["transport_mode"] = text
                        elif tag == "Line":
                            rows.append(current)
                            elem.clear()
                            break
                        elem.clear()
                    elif event == "end":
                        elem.clear()

    print(f"\nDone — {len(rows)} NeTEx lines extracted.")
    return pd.DataFrame(rows)

netex_lines = extract_all_netex_lines(netex_zip_path, line_files)
display(netex_lines.head())

  [7011/7011] line_97887.xmll
Done — 7011 NeTEx lines extracted.


,file,line_id,name,public_code,transport_mode
0,line_100935.xml,FSR:Line:100935,Mietoinen-Askainen-Lemu,127,bus
1,line_101185.xml,FSR:Line:101185,Vuosnainen - Kustavi - Turku,125,bus
2,line_101186.xml,FSR:Line:101186,Turku - Kustavi - Vuosnainen,125,bus
3,line_101193.xml,FSR:Line:101193,Tolkkinen - Sairaala,1,bus
4,line_101194.xml,FSR:Line:101194,Sairaala - Tolkkinen,1,bus


### Comment

The full extraction produced 7,011 NeTEx lines

In [61]:
# Quality check for extracted NeTEx lines

netex_lines_quality = pd.DataFrame({
    "metric": [
        "NeTEx line rows",
        "unique line IDs",
        "unique PublicCodes",
        "missing line IDs",
        "missing names",
        "missing PublicCodes",
        "missing transport modes",
        "duplicate line IDs"
    ],
    "value": [
        len(netex_lines),
        netex_lines["line_id"].nunique(),
        netex_lines["public_code"].nunique(),
        netex_lines["line_id"].isna().sum(),
        netex_lines["name"].isna().sum(),
        netex_lines["public_code"].isna().sum(),
        netex_lines["transport_mode"].isna().sum(),
        netex_lines["line_id"].duplicated().sum()
    ]
})

netex_lines_quality

,metric,value
0,NeTEx line rows,7011
1,unique line IDs,7011
2,unique PublicCodes,1875
3,missing line IDs,0
4,missing names,0
5,missing PublicCodes,1798
6,missing transport modes,13
7,duplicate line IDs,0


## Comment

The NeTEx line extraction is clean and structurally consistent.

All 7,011 lines have unique technical line IDs and no missing names.

However, there are only 1,875 unique PublicCodes and 1,798 lines have no PublicCode at all.

This shows that many technical NeTEx lines either:
- share the same public-facing line label,
- or do not expose a public code.

So the PublicCode field appears to represent the public line identifier more accurately than the technical line ID.

This supports using:
GTFS `route_short_name`
vs
NeTEx `PublicCode`
for the line-level comparison.

 ## Inspect duplicate PublicCodes

As observed above many NeTEx lines appear to share the same PublicCode.

The next step is to measure how often this happens and inspect example cases.

In [62]:
# Count how many lines share the same PublicCode

netex_publiccode_counts = (
    netex_lines[
        netex_lines["public_code"].notna()
    ]
    .groupby("public_code")
    .size()
    .reset_index(name="n_lines")
    .sort_values("n_lines", ascending=False)
)

netex_publiccode_counts.head(20)

,public_code,n_lines
1287,FlixBus,772
364,3,72
0,1,67
223,2,58
1286,Finferries,47
485,4,44
613,5,40
1,10,30
941,8,28
819,7,28


## Comment

Many NeTEx line rows share the same PublicCode. For example:

- FlixBus appears in 772 NeTEx line rows

- line 3 appears in 72 NeTEx line rows

- line 1 appears in 67 NeTEx line rows
    

This means that one visible line label can appear in many technical NeTEx line records.

So for the line-level comparison, I will compare unique public labels, not raw line rows.

All of this is consistent with the GTFS side, where several route rows can also share the same visible line label.

## Prepare NeTEx deduplicated line table

For the later line level comparison, I need one row per visible public line label.

This means I will deduplicate the line table based on `public_code`.

In [63]:
# Prepare NeTEx line labels for comparison

netex_lines_work = netex_lines.copy()

netex_lines_work["public_code_clean"] = (
    netex_lines_work["public_code"]
    .astype("string")
    .str.strip()
)

# Keep only rows with a PublicCode
netex_lines_with_code = netex_lines_work[
    netex_lines_work["public_code_clean"].notna()
].copy()

# One row per public code
netex_line_labels = (
    netex_lines_with_code
    .groupby("public_code_clean")
    .agg(
        n_line_rows=("line_id", "count"),
        transport_modes=("transport_mode", lambda x: sorted(x.dropna().unique().tolist())),
        example_line_id=("line_id", "first"),
        example_name=("name", "first")
    )
    .reset_index()
    .rename(columns={"public_code_clean": "netex_line_label"})
    .sort_values("netex_line_label")
)

netex_line_labels.head(10)

,netex_line_label,n_line_rows,transport_modes,example_line_id,example_name
0,1,67,"[bus, tram]",FSR:Line:101193,Tolkkinen - Sairaala
1,10,30,"[bus, tram]",FSR:Line:115199,Kaukajärvi - Keskustori - Tahmela
2,100,7,[bus],FSR:Line:20880,Keskusta-Turkuhalli
3,1001,1,[bus],FSR:Line:22097,P1 Kiiskinmäki
4,1003,1,[bus],FSR:Line:22098,P3 Kivisalmi
5,101,6,[bus],FSR:Line:101201,Niittylahti-Joensuu-Ylämylly
6,1011,1,[bus],FSR:Line:22100,SaiPa Yliopisto-Keskusta-Jäähalli
7,1012,1,[bus],FSR:Line:22101,SaiPa Kivisalmi-Keskusta-Jäähalli
8,102,3,[bus],FSR:Line:101202,Joensuu-Honkalampi-Liperi
9,102N,1,[bus],FSR:Line:132522,Kamppi-Lauttasaari(M)-Westend-Tapiola(M)-Teekk...


In [64]:
# Quality check for deduplicated NeTEx line labels

netex_line_labels_quality = pd.DataFrame({
    "metric": [
        "unique NeTEx line labels",
        "missing NeTEx line labels",
        "duplicate NeTEx line labels after deduplication",
        "line labels with more than one line row"
    ],
    "value": [
        len(netex_line_labels),
        netex_line_labels["netex_line_label"].isna().sum(),
        netex_line_labels["netex_line_label"].duplicated().sum(),
        (netex_line_labels["n_line_rows"] > 1).sum()
    ]
})

netex_line_labels_quality

,metric,value
0,unique NeTEx line labels,1875
1,missing NeTEx line labels,0
2,duplicate NeTEx line labels after deduplication,0
3,line labels with more than one line row,805


## Comment

- After deduplication Finland has 1,875 unique line labels (`PublicCode`)

- 805 labels correspond to more than one technical line row

## Inspect NeTEx lines without PublicCode

Some NeTEx lines do not have a PublicCode.

Before finishing the NeTEx line exploration, I inspect these rows to understand what kind of lines they are.


In [65]:
# Inspect NeTEx lines missing PublicCode

netex_lines_missing_public_code = netex_lines[
    netex_lines["public_code"].isna()
].copy()

netex_lines_missing_public_code[[
    "file",
    "line_id",
    "name",
    "public_code",
    "transport_mode"
]].head(20)

,file,line_id,name,public_code,transport_mode
99,line_106033.xml,FSR:Line:106033,Kolinportti - Koli,None,bus
100,line_106034.xml,FSR:Line:106034,Koli - Kolinportti,None,bus
119,line_109468.xml,FSR:Line:109468,Kuhmo - Tiilikangas - Sotkamo - Kajaani,None,bus
124,line_110453.xml,FSR:Line:110453,Kajaani - Sotkamo - Tiilikangas,None,bus
125,line_110454.xml,FSR:Line:110454,Tiilikangas - Sotkamo - Kajaani,None,bus
126,line_110480.xml,FSR:Line:110480,Saarijärvi - Äänekoski - Jyväskylä,None,bus
127,line_110481.xml,FSR:Line:110481,Jyväskylä - Äänekoski - Saarijärvi,None,bus
128,line_110498.xml,FSR:Line:110498,Rasua - Puistola - Jämsänkoski,None,bus
129,line_110499.xml,FSR:Line:110499,Jämsänkoski - Puistola - Rasua,None,bus
130,line_110500.xml,FSR:Line:110500,Jämsä - Jämsä ras - Jämsänkoski,None,bus


In [66]:
# Count missing PublicCode rows by transport mode

netex_missing_public_code_by_mode = (
    netex_lines_missing_public_code["transport_mode"]
    .fillna("missing")
    .value_counts(dropna=False)
    .reset_index()
)

netex_missing_public_code_by_mode.columns = [
    "transport_mode",
    "n_lines_missing_public_code"
]

netex_missing_public_code_by_mode

,transport_mode,n_lines_missing_public_code
0,bus,1684
1,water,110
2,missing,4


## Inspect NeTEx transport modes

I also inspect the transport modes present in the NeTEx line table.


In [67]:
# Count NeTEx line rows by transport mode

netex_transport_modes = (
    netex_lines["transport_mode"]
    .fillna("missing")
    .value_counts(dropna=False)
    .reset_index()
)

netex_transport_modes.columns = ["transport_mode", "n_line_rows"]

netex_transport_modes

,transport_mode,n_line_rows
0,bus,5867
1,rail,698
2,air,224
3,water,175
4,tram,30
5,missing,13
6,metro,4


## Comment


- Most of the lines without a PublicCode are bus lines, and a smaller number are water lines

- The NeTEx line table also contains several transport modes: bus, rail, air, water, tram, and metro


## Inspect NeTEx calendar references in line files

Before extracting calendar data from the large `_shared_data.xml` file, I first inspect a few line files.

The goal is to see how `ServiceJourney` elements refer to calendar information.

In [68]:
# Inspect calendar-related tags in a few line files

calendar_related_tags = [
    "ServiceJourney",
    "DayTypeRef",
    "OperatingPeriodRef",
    "AvailabilityConditionRef",
    "ValidDayBits",
    "UicOperatingPeriod",
    "FromDate",
    "ToDate"
]

sample_line_files = line_files[:10]

netex_line_calendar_tag_checks = []

for xml_file in sample_line_files:
    check = count_target_tags_in_xml(
        netex_zip_path,
        xml_file,
        target_tags=calendar_related_tags,
        max_events=None
    )
    netex_line_calendar_tag_checks.append(check)

netex_line_calendar_tag_check = pd.concat(
    netex_line_calendar_tag_checks,
    ignore_index=True
)

netex_line_calendar_tag_check

,file,tag,count_in_scan,scanned_events,scan_limited
0,line_100935.xml,ServiceJourney,2,337,False
1,line_100935.xml,DayTypeRef,2,337,False
2,line_100935.xml,OperatingPeriodRef,0,337,False
3,line_100935.xml,AvailabilityConditionRef,0,337,False
4,line_100935.xml,ValidDayBits,0,337,False
...,...,...,...,...,...
75,line_101199.xml,AvailabilityConditionRef,0,884,False
76,line_101199.xml,ValidDayBits,0,884,False
77,line_101199.xml,UicOperatingPeriod,0,884,False
78,line_101199.xml,FromDate,0,884,False


## Comment

The sampled line files contain `ServiceJourney` and `DayTypeRef`.


The line files do not contain `ValidDayBits`, `UicOperatingPeriod`, `FromDate`, or `ToDate`.

From the output the structure seems to be:
    
- line_*.xml
ServiceJourney → DayTypeRef

- _shared_data.xml
probably contains the DayType definitions

## Inspect sample ServiceJourney DayTypeRefs

The tag check showed that ServiceJourneys refer to DayTypes.

Now I extract a small sample of ServiceJourney IDs and their DayTypeRef values from the line files.

In [69]:
def extract_servicejourney_daytype_sample(zip_path, line_files, n_files=5, max_rows=30):
    # Store sample rows
    rows = []

    # Open the NeTEx ZIP once
    with zipfile.ZipFile(zip_path, "r") as z:

        # Loop through a few line files
        for xml_file in line_files[:n_files]:

            with z.open(xml_file) as f:
                current = None
                inside_service_journey = False

                # Parse XML step by step
                for event, elem in ET.iterparse(f, events=("start", "end")):
                    tag = local_name(elem.tag)

                    # Start of ServiceJourney
                    if event == "start" and tag == "ServiceJourney":
                        inside_service_journey = True
                        current = {
                            "file": xml_file,
                            "service_journey_id": elem.attrib.get("id"),
                            "daytype_ref": None
                        }

                    # Read DayTypeRef inside ServiceJourney
                    elif event == "end" and inside_service_journey and tag == "DayTypeRef":
                        if current is not None:
                            current["daytype_ref"] = elem.attrib.get("ref")
                        elem.clear()

                    # End of ServiceJourney
                    elif event == "end" and tag == "ServiceJourney":
                        if current is not None:
                            rows.append(current)

                        current = None
                        inside_service_journey = False
                        elem.clear()

                        if len(rows) >= max_rows:
                            return pd.DataFrame(rows)

                    elif event == "end":
                        elem.clear()

    return pd.DataFrame(rows)

netex_servicejourney_daytype_sample = extract_servicejourney_daytype_sample(
    netex_zip_path,
    line_files,
    n_files=5,
    max_rows=30
)

netex_servicejourney_daytype_sample

,file,service_journey_id,daytype_ref
0,line_100935.xml,VAR:ServiceJourney:T2026-BR-VS-M-P_1273_0_0625...,VAR:DayType:T2026-BR-VS-M-P
1,line_100935.xml,VAR:ServiceJourney:T2026-BR-VS-M-P_1273_1_1710...,VAR:DayType:T2026-BR-VS-M-P
2,line_101185.xml,OYM:ServiceJourney:207805_474,OYM:DayType:207805.474
3,line_101186.xml,OYM:ServiceJourney:226124_474,OYM:DayType:226124.474
4,line_101186.xml,OYM:ServiceJourney:226125_474,OYM:DayType:226125.474
5,line_101193.xml,OYM:ServiceJourney:235060_7010,OYM:DayType:235060.7010
6,line_101193.xml,OYM:ServiceJourney:235061_7010,OYM:DayType:235061.7010
7,line_101193.xml,OYM:ServiceJourney:235073_7010,OYM:DayType:235073.7010
8,line_101193.xml,OYM:ServiceJourney:235074_7010,OYM:DayType:235074.7010
9,line_101193.xml,OYM:ServiceJourney:236358_7010,OYM:DayType:236358.7010


## Comment

The sample shows that ServiceJourney elements contain DayTypeRef values.

This means that service journeys are linked to calendar information through DayType IDs.

The actual calendar definitions are not inside the line file sample.  
They are probably stored in `_shared_data.xml`.

So the next step would be to inspect `_shared_data.xml` and find where these DayType IDs are defined.

## Search calendar keywords in shared data

Because the `_shared_data.xml` file is very large I first check whether important calendar-related words appear in the file text.

This is a quick way to confirm whether `_shared_data.xml` contains DayType or calendar information.

In [70]:
# Quick text search for calendar-related keywords in _shared_data.xml

calendar_keywords = [
    b"DayType",
    b"DayTypeAssignment",
    b"OperatingPeriod",
    b"UicOperatingPeriod",
    b"ValidDayBits",
    b"AvailabilityCondition",
    b"FromDate",
    b"ToDate"
]

keyword_counts = {keyword.decode("utf-8"): 0 for keyword in calendar_keywords}

with zipfile.ZipFile(netex_zip_path, "r") as z:
    with z.open("_shared_data.xml") as f:
        while True:
            chunk = f.read(1024 * 1024)  # read 1 MB at a time
            if not chunk:
                break

            for keyword in calendar_keywords:
                keyword_counts[keyword.decode("utf-8")] += chunk.count(keyword)

shared_calendar_keyword_check = pd.DataFrame({
    "keyword": list(keyword_counts.keys()),
    "count_in_file_text": list(keyword_counts.values())
})

shared_calendar_keyword_check

,keyword,count_in_file_text
0,DayType,92894
1,DayTypeAssignment,0
2,OperatingPeriod,60647
3,UicOperatingPeriod,0
4,ValidDayBits,0
5,AvailabilityCondition,2
6,FromDate,360
7,ToDate,360


## Comment

The keyword search confirms that `_shared_data.xml` contains calendar-related information.

There are many occurrences of `DayType` and `OperatingPeriod`.

However, `DayTypeAssignment`, `ValidDayBits`, and `UicOperatingPeriod` do not appear.

The next step is to inspect actual `DayType` and `OperatingPeriod` elements from `_shared_data.xml` to understand how active dates are represented.

## Inspect sample DayType and OperatingPeriod elements


In [71]:
def extract_shared_calendar_sample(zip_path, xml_file, n_daytypes=10, n_operating_periods=10):
    daytype_rows = []
    operating_period_rows = []
    events_scanned = 0

    with zipfile.ZipFile(zip_path, "r") as z:
        with z.open(xml_file) as f:
            for event, elem in ET.iterparse(f, events=("end",)):
                tag = local_name(elem.tag)
                events_scanned += 1

                if events_scanned % 100000 == 0:
                    print(
                        f"  {events_scanned:,} events scanned — "
                        f"{len(daytype_rows)}/{n_daytypes} DayTypes, "
                        f"{len(operating_period_rows)}/{n_operating_periods} OperatingPeriods",
                        end="\r"
                    )

                if tag == "DayType" and len(daytype_rows) < n_daytypes:
                    row = {
                        "daytype_id":           elem.attrib.get("id"),
                        "name":                 None,
                        "operating_period_ref": None
                    }
                    for child in elem.iter():
                        child_tag = local_name(child.tag)
                        text = child.text.strip() if child.text else None
                        if child_tag == "Name" and row["name"] is None:
                            row["name"] = text
                        elif child_tag == "OperatingPeriodRef" and row["operating_period_ref"] is None:
                            row["operating_period_ref"] = child.attrib.get("ref")
                    daytype_rows.append(row)
                    elem.clear()

                elif tag == "OperatingPeriod" and len(operating_period_rows) < n_operating_periods:
                    row = {
                        "operating_period_id": elem.attrib.get("id"),
                        "from_date":           None,
                        "to_date":             None
                    }
                    for child in elem.iter():
                        child_tag = local_name(child.tag)
                        text = child.text.strip() if child.text else None
                        if child_tag == "FromDate" and row["from_date"] is None:
                            row["from_date"] = text
                        elif child_tag == "ToDate" and row["to_date"] is None:
                            row["to_date"] = text
                    operating_period_rows.append(row)
                    elem.clear()

                else:
                    elem.clear()

                if (
                    len(daytype_rows) >= n_daytypes
                    and len(operating_period_rows) >= n_operating_periods
                ):
                    break

    print(f"\nDone — {len(daytype_rows)} DayTypes, {len(operating_period_rows)} OperatingPeriods sampled.")
    return (
        pd.DataFrame(daytype_rows),
        pd.DataFrame(operating_period_rows)
    )

netex_daytype_sample, netex_operating_period_sample = extract_shared_calendar_sample(
    netex_zip_path,
    "_shared_data.xml",
    n_daytypes=10,
    n_operating_periods=10
)
display(netex_daytype_sample)
display(netex_operating_period_sample)

  31,900,000 events scanned — 10/10 DayTypes, 0/10 OperatingPeriods
Done — 10 DayTypes, 10 OperatingPeriods sampled.


,daytype_id,name,operating_period_ref
0,ALA:DayType:1,None,None
1,ALA:DayType:10,None,None
2,ALA:DayType:100,None,None
3,ALA:DayType:101,None,None
4,ALA:DayType:102,None,None
5,ALA:DayType:103,None,None
6,ALA:DayType:104,None,None
7,ALA:DayType:105,None,None
8,ALA:DayType:106,None,None
9,ALA:DayType:107,None,None


,operating_period_id,from_date,to_date
0,ALA:OperatingPeriod:100,None,None
1,ALA:OperatingPeriod:101,None,None
2,ALA:OperatingPeriod:102,None,None
3,ALA:OperatingPeriod:103,None,None
4,ALA:OperatingPeriod:104,None,None
5,ALA:OperatingPeriod:105,None,None
6,ALA:OperatingPeriod:106,None,None
7,ALA:OperatingPeriod:107,None,None
8,ALA:OperatingPeriod:108,None,None
9,ALA:OperatingPeriod:109,None,None


## Comment

The first sample found `DayType` and `OperatingPeriod` IDs, but it did not extract useful date information.

The sampled elements do not show names, operating-period references, `FromDate`, or `ToDate`.

This means the calendar structure is different from the previous countries, or the useful date information is stored somewhere else in `_shared_data.xml`.


## Inspect internal structure of DayType and OperatingPeriod

The first calendar sample did not show useful date fields.

Now I inspect the child tags inside `DayType` and `OperatingPeriod` elements.  
This helps understand where the calendar information is stored.

In [72]:
def inspect_element_structure(zip_path, xml_file, target_tag, n_sample=3, max_depth=3):
    rows = []
    found = 0
    events_scanned = 0

    with zipfile.ZipFile(zip_path, "r") as z:
        with z.open(xml_file) as f:
            for event, elem in ET.iterparse(f, events=("end",)):
                tag = local_name(elem.tag)
                events_scanned += 1

                if events_scanned % 100000 == 0:
                    print(f"  {events_scanned:,} events scanned — {found}/{n_sample} {target_tag} found", end="\r")

                if tag == target_tag:
                    found += 1
                    for child in elem.iter():
                        child_tag = local_name(child.tag)
                        text = child.text.strip() if child.text and child.text.strip() else None
                        rows.append({
                            "sample_number": found,
                            "target_tag":    target_tag,
                            "element_id":    elem.attrib.get("id"),
                            "child_tag":     child_tag,
                            "child_text":    text,
                            "child_ref":     child.attrib.get("ref")
                        })
                    elem.clear()
                    if found >= n_sample:
                        break

                else:
                    elem.clear()

    print(f"\nDone — {found} {target_tag} elements inspected.")
    return pd.DataFrame(rows)

daytype_structure_sample = inspect_element_structure(
    netex_zip_path,
    "_shared_data.xml",
    target_tag="DayType",
    n_sample=3
)
operating_period_structure_sample = inspect_element_structure(
    netex_zip_path,
    "_shared_data.xml",
    target_tag="OperatingPeriod",
    n_sample=3
)

display(daytype_structure_sample.head(50))
display(operating_period_structure_sample.head(50))

  31,700,000 events scanned — 0/3 DayType found
Done — 3 DayType elements inspected.
  31,900,000 events scanned — 0/3 OperatingPeriod found
Done — 3 OperatingPeriod elements inspected.


,sample_number,target_tag,element_id,child_tag,child_text,child_ref
0,1,DayType,ALA:DayType:1,DayType,None,None
1,1,DayType,ALA:DayType:1,properties,None,None
2,2,DayType,ALA:DayType:10,DayType,None,None
3,2,DayType,ALA:DayType:10,properties,None,None
4,3,DayType,ALA:DayType:100,DayType,None,None
5,3,DayType,ALA:DayType:100,properties,None,None


,sample_number,target_tag,element_id,child_tag,child_text,child_ref
0,1,OperatingPeriod,ALA:OperatingPeriod:100,OperatingPeriod,None,None
1,1,OperatingPeriod,ALA:OperatingPeriod:100,FromOperatingDayRef,None,None
2,1,OperatingPeriod,ALA:OperatingPeriod:100,ToOperatingDayRef,None,None
3,2,OperatingPeriod,ALA:OperatingPeriod:101,OperatingPeriod,None,None
4,2,OperatingPeriod,ALA:OperatingPeriod:101,FromOperatingDayRef,None,None
5,2,OperatingPeriod,ALA:OperatingPeriod:101,ToOperatingDayRef,None,None
6,3,OperatingPeriod,ALA:OperatingPeriod:102,OperatingPeriod,None,None
7,3,OperatingPeriod,ALA:OperatingPeriod:102,FromOperatingDayRef,None,None
8,3,OperatingPeriod,ALA:OperatingPeriod:102,ToOperatingDayRef,None,None


## Comment

The internal structure shows that `DayType` itself does not contain useful date information.

`OperatingPeriod` also does not contain direct `FromDate` or `ToDate`.

Instead, `OperatingPeriod` refers to `FromOperatingDayRef` and `ToOperatingDayRef`.

So the structure seems to be:

OperatingPeriod
→ FromOperatingDayRef
→ ToOperatingDayRef

This means Finland stores calendar validity through OperatingDay references.

The next step is to inspect `OperatingDay` elements and check whether they contain the actual dates.

## Inspect OperatingDay structure

The previous check showed that `OperatingPeriod` refers to `OperatingDay` elements.

Now I inspect `OperatingDay` elements to see whether they contain the actual calendar dates.

In [73]:
# Search for OperatingDay-related keywords in _shared_data.xml
operating_day_keywords = [
    b"OperatingDay",
    b"FromOperatingDayRef",
    b"ToOperatingDayRef",
    b"CalendarDate",
    b"Date"
]
operating_day_keyword_counts = {
    keyword.decode("utf-8"): 0
    for keyword in operating_day_keywords
}
bytes_read = 0
with zipfile.ZipFile(netex_zip_path, "r") as z:
    with z.open("_shared_data.xml") as f:
        while True:
            chunk = f.read(1024 * 1024)
            if not chunk:
                break
            bytes_read += len(chunk)
            print(f"  {bytes_read / 1024**2:.1f} MB scanned...", end="\r")
            for keyword in operating_day_keywords:
                operating_day_keyword_counts[keyword.decode("utf-8")] += chunk.count(keyword)

print(f"\nDone — {bytes_read / 1024**2:.1f} MB scanned.")
operating_day_keyword_check = pd.DataFrame({
    "keyword":             list(operating_day_keyword_counts.keys()),
    "count_in_file_text":  list(operating_day_keyword_counts.values())
})
display(operating_day_keyword_check)

  1179.7 MB scanned...
Done — 1179.7 MB scanned.


,keyword,count_in_file_text
0,OperatingDay,173996
1,FromOperatingDayRef,20046
2,ToOperatingDayRef,20046
3,CalendarDate,62556
4,Date,63276


In [74]:
# Inspect OperatingDay elements and keep all attributes
def inspect_element_structure_with_attributes(zip_path, xml_file, target_tag, n_sample=5):
    rows = []
    found = 0
    events_scanned = 0

    with zipfile.ZipFile(zip_path, "r") as z:
        with z.open(xml_file) as f:
            for event, elem in ET.iterparse(f, events=("end",)):
                tag = local_name(elem.tag)
                events_scanned += 1

                if events_scanned % 100000 == 0:
                    print(f"  {events_scanned:,} events scanned — {found}/{n_sample} {target_tag} found", end="\r")

                if tag == target_tag:
                    found += 1
                    for child in elem.iter():
                        child_tag = local_name(child.tag)
                        text = child.text.strip() if child.text and child.text.strip() else None
                        rows.append({
                            "sample_number":    found,
                            "target_tag":       target_tag,
                            "element_id":       elem.attrib.get("id"),
                            "child_tag":        child_tag,
                            "child_text":       text,
                            "child_attributes": dict(child.attrib)
                        })
                    elem.clear()
                    if found >= n_sample:
                        break
                else:
                    elem.clear()

    print(f"\nDone — {found} {target_tag} elements inspected.")
    return pd.DataFrame(rows)

operating_day_structure_sample = inspect_element_structure_with_attributes(
    netex_zip_path,
    "_shared_data.xml",
    target_tag="OperatingDay",
    n_sample=5
)
display(operating_day_structure_sample.head(80))

  31,800,000 events scanned — 0/5 OperatingDay found
Done — 5 OperatingDay elements inspected.


,sample_number,target_tag,element_id,child_tag,child_text,child_attributes
0,1,OperatingDay,ALA:OperatingDay:2026-04-23,OperatingDay,None,"{'version': '1', 'id': 'ALA:OperatingDay:2026-..."
1,1,OperatingDay,ALA:OperatingDay:2026-04-23,CalendarDate,None,{}
2,2,OperatingDay,ALA:OperatingDay:2026-04-28,OperatingDay,None,"{'version': '1', 'id': 'ALA:OperatingDay:2026-..."
3,2,OperatingDay,ALA:OperatingDay:2026-04-28,CalendarDate,None,{}
4,3,OperatingDay,ALA:OperatingDay:2026-04-29,OperatingDay,None,"{'version': '1', 'id': 'ALA:OperatingDay:2026-..."
5,3,OperatingDay,ALA:OperatingDay:2026-04-29,CalendarDate,None,{}
6,4,OperatingDay,ALA:OperatingDay:2026-04-30,OperatingDay,None,"{'version': '1', 'id': 'ALA:OperatingDay:2026-..."
7,4,OperatingDay,ALA:OperatingDay:2026-04-30,CalendarDate,None,{}
8,5,OperatingDay,ALA:OperatingDay:2026-05-01,OperatingDay,None,"{'version': '1', 'id': 'ALA:OperatingDay:2026-..."
9,5,OperatingDay,ALA:OperatingDay:2026-05-01,CalendarDate,None,{}


In [75]:
# Inspect OperatingPeriod again, including all reference attributes

operating_period_structure_with_attributes = inspect_element_structure_with_attributes(
    netex_zip_path,
    "_shared_data.xml",
    target_tag="OperatingPeriod",
    n_sample=5
)

display(operating_period_structure_with_attributes.head(80))

  31,900,000 events scanned — 0/5 OperatingPeriod found
Done — 5 OperatingPeriod elements inspected.


,sample_number,target_tag,element_id,child_tag,child_text,child_attributes
0,1,OperatingPeriod,ALA:OperatingPeriod:100,OperatingPeriod,None,"{'version': '1', 'id': 'ALA:OperatingPeriod:100'}"
1,1,OperatingPeriod,ALA:OperatingPeriod:100,FromOperatingDayRef,None,{}
2,1,OperatingPeriod,ALA:OperatingPeriod:100,ToOperatingDayRef,None,{}
3,2,OperatingPeriod,ALA:OperatingPeriod:101,OperatingPeriod,None,"{'version': '1', 'id': 'ALA:OperatingPeriod:101'}"
4,2,OperatingPeriod,ALA:OperatingPeriod:101,FromOperatingDayRef,None,{}
5,2,OperatingPeriod,ALA:OperatingPeriod:101,ToOperatingDayRef,None,{}
6,3,OperatingPeriod,ALA:OperatingPeriod:102,OperatingPeriod,None,"{'version': '1', 'id': 'ALA:OperatingPeriod:102'}"
7,3,OperatingPeriod,ALA:OperatingPeriod:102,FromOperatingDayRef,None,{}
8,3,OperatingPeriod,ALA:OperatingPeriod:102,ToOperatingDayRef,None,{}
9,4,OperatingPeriod,ALA:OperatingPeriod:103,OperatingPeriod,None,"{'version': '1', 'id': 'ALA:OperatingPeriod:103'}"


## Comment


The keyword search confirms that `_shared_data.xml` contains OperatingDay elements and CalendarDate information.

The structure seems to be different from other countries seen before.

Instead of direct `FromDate` and `ToDate`, Finland uses:

OperatingPeriod  
→ FromOperatingDayRef  
→ ToOperatingDayRef

The earlier inspection probably didn't show the references clearly, probably because the XML elements were cleared too early during parsing.

So the next step is to extract OperatingDay and OperatingPeriod samples with a more careful parser.

## Extract sample OperatingDays and OperatingPeriods carefully

The previous structure inspection showed that OperatingPeriods refer to OperatingDays.

Now I extract samples more carefully, without clearing child elements too early.

This should show whether OperatingDay IDs contain the date and whether OperatingPeriod gives start and end OperatingDay references.

In [76]:
def extract_operatingday_sample(zip_path, xml_file, n_sample=10):
    rows = []
    current = None
    depth = 0
    events_scanned = 0

    with zipfile.ZipFile(zip_path, "r") as z:
        with z.open(xml_file) as f:
            for event, elem in ET.iterparse(f, events=("start", "end")):
                tag = local_name(elem.tag)
                events_scanned += 1

                if events_scanned % 100000 == 0:
                    print(f"  {events_scanned:,} events scanned — {len(rows)}/{n_sample} OperatingDays found", end="\r")

                if event == "start" and tag == "OperatingDay":
                    depth += 1
                    if depth == 1:
                        current = {
                            "operating_day_id": elem.attrib.get("id"),
                            "calendar_date":    None
                        }

                elif event == "end" and tag == "OperatingDay":
                    if depth == 1 and current is not None:
                        rows.append(current)
                        current = None
                    depth -= 1
                    elem.clear()
                    if len(rows) >= n_sample:
                        break

                elif event == "end" and current is not None and depth > 0:
                    if tag == "CalendarDate" and current["calendar_date"] is None:
                        current["calendar_date"] = elem.text.strip() if elem.text else None
                    elem.clear()

                elif event == "end":
                    elem.clear()

    print(f"\nDone — {len(rows)} OperatingDay elements sampled.")
    return pd.DataFrame(rows)

netex_operatingday_sample = extract_operatingday_sample(
    netex_zip_path,
    "_shared_data.xml",
    n_sample=10
)
display(netex_operatingday_sample)

  63,600,000 events scanned — 0/10 OperatingDays found
Done — 10 OperatingDay elements sampled.


,operating_day_id,calendar_date
0,ALA:OperatingDay:2026-04-23,2026-04-23
1,ALA:OperatingDay:2026-04-28,2026-04-28
2,ALA:OperatingDay:2026-04-29,2026-04-29
3,ALA:OperatingDay:2026-04-30,2026-04-30
4,ALA:OperatingDay:2026-05-01,2026-05-01
5,ALA:OperatingDay:2026-05-02,2026-05-02
6,ALA:OperatingDay:2026-05-03,2026-05-03
7,ALA:OperatingDay:2026-05-04,2026-05-04
8,ALA:OperatingDay:2026-05-05,2026-05-05
9,ALA:OperatingDay:2026-05-06,2026-05-06


In [77]:
def extract_operatingperiod_sample(zip_path, xml_file, n_sample=10):
    rows = []
    current = None
    depth = 0
    events_scanned = 0

    with zipfile.ZipFile(zip_path, "r") as z:
        with z.open(xml_file) as f:
            for event, elem in ET.iterparse(f, events=("start", "end")):
                tag = local_name(elem.tag)
                events_scanned += 1

                if events_scanned % 100000 == 0:
                    print(f"  {events_scanned:,} events scanned — {len(rows)}/{n_sample} OperatingPeriods found", end="\r")

                if event == "start" and tag == "OperatingPeriod":
                    depth += 1
                    if depth == 1:
                        current = {
                            "operating_period_id":    elem.attrib.get("id"),
                            "from_operating_day_ref": None,
                            "to_operating_day_ref":   None
                        }

                elif event == "end" and tag == "OperatingPeriod":
                    if depth == 1 and current is not None:
                        rows.append(current)
                        current = None
                    depth -= 1
                    elem.clear()
                    if len(rows) >= n_sample:
                        break

                elif event == "start" and current is not None and depth > 0:
                    if tag == "FromOperatingDayRef" and current["from_operating_day_ref"] is None:
                        current["from_operating_day_ref"] = (
                            elem.attrib.get("ref")
                            or elem.attrib.get("idref")
                            or elem.attrib.get("href")
                        )
                    elif tag == "ToOperatingDayRef" and current["to_operating_day_ref"] is None:
                        current["to_operating_day_ref"] = (
                            elem.attrib.get("ref")
                            or elem.attrib.get("idref")
                            or elem.attrib.get("href")
                        )

                elif event == "end":
                    elem.clear()

    print(f"\nDone — {len(rows)} OperatingPeriod elements sampled.")
    return pd.DataFrame(rows)

netex_operatingperiod_sample = extract_operatingperiod_sample(
    netex_zip_path,
    "_shared_data.xml",
    n_sample=10
)
display(netex_operatingperiod_sample)

  63,800,000 events scanned — 0/10 OperatingPeriods found
Done — 10 OperatingPeriod elements sampled.


,operating_period_id,from_operating_day_ref,to_operating_day_ref
0,ALA:OperatingPeriod:100,ALA:OperatingDay:2026-04-23,ALA:OperatingDay:2027-10-14
1,ALA:OperatingPeriod:101,ALA:OperatingDay:2026-04-23,ALA:OperatingDay:2027-10-14
2,ALA:OperatingPeriod:102,ALA:OperatingDay:2026-04-23,ALA:OperatingDay:2027-10-14
3,ALA:OperatingPeriod:103,ALA:OperatingDay:2026-04-23,ALA:OperatingDay:2027-10-14
4,ALA:OperatingPeriod:104,ALA:OperatingDay:2026-04-23,ALA:OperatingDay:2027-10-14
5,ALA:OperatingPeriod:105,ALA:OperatingDay:2026-04-23,ALA:OperatingDay:2027-10-14
6,ALA:OperatingPeriod:106,ALA:OperatingDay:2026-04-23,ALA:OperatingDay:2027-10-14
7,ALA:OperatingPeriod:107,ALA:OperatingDay:2026-04-23,ALA:OperatingDay:2027-10-14
8,ALA:OperatingPeriod:108,ALA:OperatingDay:2026-04-23,ALA:OperatingDay:2027-10-14
9,ALA:OperatingPeriod:109,ALA:OperatingDay:2026-04-23,ALA:OperatingDay:2027-10-14


## Comment 

From the output it seems that Finland uses the following structure:
    
OperatingPeriod
→ FromOperatingDayRef
→ ToOperatingDayRef
→ OperatingDay
→ CalendarDate


- `OperatingDay` elements contain the actual calendar dates, for example 2026-04-23

- `OperatingPeriod` elements do not contain dates directly
Instead, they refer to a start OperatingDay and an end OperatingDay

- So Finland stores calendar validity through OperatingDay references, not through direct FromDate and ToDate fields



## Inspect DayTypes used by ServiceJourneys

The line files show that ServiceJourney elements refer to DayType IDs.

Now I inspect some of these exact DayType IDs inside `_shared_data.xml`.


The goal to see whether the used `DayType` elements contain something like:

- `OperatingPeriodRef`

- `PropertyOfDay`

- `DayTypeAssignment`

In [78]:
# Select some DayTypeRefs that are actually used by ServiceJourneys

sample_daytype_refs_used = (
    netex_servicejourney_daytype_sample["daytype_ref"]
    .dropna()
    .drop_duplicates()
    .head(10)
    .tolist()
)

sample_daytype_refs_used

['VAR:DayType:T2026-BR-VS-M-P',
 'OYM:DayType:207805.474',
 'OYM:DayType:226124.474',
 'OYM:DayType:226125.474',
 'OYM:DayType:235060.7010',
 'OYM:DayType:235061.7010',
 'OYM:DayType:235073.7010',
 'OYM:DayType:235074.7010',
 'OYM:DayType:236358.7010',
 'OYM:DayType:236359.7010']

In [79]:
def inspect_specific_daytypes(zip_path, xml_file, target_daytype_ids):
    rows = []
    target_daytype_ids = set(target_daytype_ids)
    inside_target = False
    current_daytype_id = None
    depth = 0
    found_ids = set()
    events_scanned = 0

    with zipfile.ZipFile(zip_path, "r") as z:
        with z.open(xml_file) as f:
            for event, elem in ET.iterparse(f, events=("start", "end")):
                tag = local_name(elem.tag)
                events_scanned += 1

                if events_scanned % 100000 == 0:
                    print(
                        f"  {events_scanned:,} events scanned — "
                        f"{len(found_ids)}/{len(target_daytype_ids)} DayTypes found",
                        end="\r"
                    )

                if (
                    event == "start"
                    and tag == "DayType"
                    and elem.attrib.get("id") in target_daytype_ids
                    and not inside_target
                ):
                    inside_target = True
                    current_daytype_id = elem.attrib.get("id")
                    depth = 1
                    rows.append({
                        "daytype_id": current_daytype_id,
                        "event":      "start",
                        "tag":        tag,
                        "text":       None,
                        "attributes": dict(elem.attrib)
                    })

                elif event == "start" and inside_target:
                    depth += 1
                    rows.append({
                        "daytype_id": current_daytype_id,
                        "event":      "start",
                        "tag":        tag,
                        "text":       None,
                        "attributes": dict(elem.attrib)
                    })

                elif event == "end" and inside_target:
                    text = elem.text.strip() if elem.text and elem.text.strip() else None
                    rows.append({
                        "daytype_id": current_daytype_id,
                        "event":      "end",
                        "tag":        tag,
                        "text":       text,
                        "attributes": dict(elem.attrib)
                    })
                    if tag == "DayType" and depth == 1:
                        found_ids.add(current_daytype_id)
                        inside_target = False
                        current_daytype_id = None
                        depth = 0
                        elem.clear()
                        if found_ids == target_daytype_ids:
                            break
                    else:
                        depth -= 1
                        elem.clear()

                elif event == "end":
                    elem.clear()

    print(f"\nDone — {len(found_ids)} out of {len(target_daytype_ids)} DayTypes found.")
    return pd.DataFrame(rows)

specific_daytype_structure = inspect_specific_daytypes(
    netex_zip_path,
    "_shared_data.xml",
    sample_daytype_refs_used
)
display(specific_daytype_structure.head(80))

  63,600,000 events scanned — 0/10 DayTypes found
Done — 10 out of 10 DayTypes found.


,daytype_id,event,tag,text,attributes
0,OYM:DayType:207805.474,start,DayType,None,"{'version': '1', 'id': 'OYM:DayType:207805.474'}"
1,OYM:DayType:207805.474,end,DayType,None,"{'version': '1', 'id': 'OYM:DayType:207805.474'}"
2,OYM:DayType:226124.474,start,DayType,None,"{'version': '1', 'id': 'OYM:DayType:226124.474'}"
3,OYM:DayType:226124.474,end,DayType,None,"{'version': '1', 'id': 'OYM:DayType:226124.474'}"
4,OYM:DayType:226125.474,start,DayType,None,"{'version': '1', 'id': 'OYM:DayType:226125.474'}"
5,OYM:DayType:226125.474,end,DayType,None,"{'version': '1', 'id': 'OYM:DayType:226125.474'}"
6,OYM:DayType:235060.7010,start,DayType,None,"{'version': '1', 'id': 'OYM:DayType:235060.7010'}"
7,OYM:DayType:235060.7010,end,DayType,None,"{'version': '1', 'id': 'OYM:DayType:235060.7010'}"
8,OYM:DayType:235061.7010,start,DayType,None,"{'version': '1', 'id': 'OYM:DayType:235061.7010'}"
9,OYM:DayType:235061.7010,end,DayType,None,"{'version': '1', 'id': 'OYM:DayType:235061.7010'}"


## Comment

The selected DayTypes were found in `_shared_data.xml`.

Most of the sampled DayTypes only contain the DayType ID and no extra calendar information.

One DayType, `VAR:DayType:T2026-BR-VS-M-P`, contains a `DaysOfWeek` value:

Monday Tuesday Wednesday Thursday Friday

This means that some Finnish NeTEx DayTypes store a weekday pattern directly inside the DayType.

However, many other DayTypes are empty in this structure check.

So Finland’s calendar structure is mixed:
some DayTypes contain weekday information, while others may depend on other references or naming logic.

## Check whether DayTypes are linked to OperatingPeriods by ID

Some DayTypes used by ServiceJourneys do not contain weekday or date information directly.

Now I check whether these DayType IDs have matching OperatingPeriod IDs in `_shared_data.xml`.

This helps find the missing link between:

ServiceJourney → DayTypeRef → calendar validity


The `goal` is to see:

If `candidate_operating_period_count` > 0,

then the `DayType` may be linked to an OperatingPeriod by the same ID pattern

In [80]:
# Create possible matching OperatingPeriod IDs from the sampled DayTypeRefs

daytype_operating_period_candidates = pd.DataFrame({
    "daytype_ref": sample_daytype_refs_used
})

daytype_operating_period_candidates["candidate_operating_period_id"] = (
    daytype_operating_period_candidates["daytype_ref"]
    .str.replace(":DayType:", ":OperatingPeriod:", regex=False)
)

daytype_operating_period_candidates

,daytype_ref,candidate_operating_period_id
0,VAR:DayType:T2026-BR-VS-M-P,VAR:OperatingPeriod:T2026-BR-VS-M-P
1,OYM:DayType:207805.474,OYM:OperatingPeriod:207805.474
2,OYM:DayType:226124.474,OYM:OperatingPeriod:226124.474
3,OYM:DayType:226125.474,OYM:OperatingPeriod:226125.474
4,OYM:DayType:235060.7010,OYM:OperatingPeriod:235060.7010
5,OYM:DayType:235061.7010,OYM:OperatingPeriod:235061.7010
6,OYM:DayType:235073.7010,OYM:OperatingPeriod:235073.7010
7,OYM:DayType:235074.7010,OYM:OperatingPeriod:235074.7010
8,OYM:DayType:236358.7010,OYM:OperatingPeriod:236358.7010
9,OYM:DayType:236359.7010,OYM:OperatingPeriod:236359.7010


In [81]:
# Count whether the candidate OperatingPeriod IDs appear in _shared_data.xml

def count_terms_in_zip_text(zip_path, xml_file, terms):
    # Store counts for each search term
    counts = {term: 0 for term in terms}

    # Convert terms to bytes
    terms_bytes = {term: term.encode("utf-8") for term in terms}

    # Open ZIP and scan file in chunks
    with zipfile.ZipFile(zip_path, "r") as z:
        with z.open(xml_file) as f:
            while True:
                chunk = f.read(1024 * 1024)

                if not chunk:
                    break

                for term, term_bytes in terms_bytes.items():
                    counts[term] += chunk.count(term_bytes)

    return counts

terms_to_check = (
    daytype_operating_period_candidates["daytype_ref"].tolist()
    + daytype_operating_period_candidates["candidate_operating_period_id"].tolist()
)

term_counts = count_terms_in_zip_text(
    netex_zip_path,
    "_shared_data.xml",
    terms_to_check
)

daytype_operating_period_candidates["daytype_ref_count"] = (
    daytype_operating_period_candidates["daytype_ref"].map(term_counts)
)

daytype_operating_period_candidates["candidate_operating_period_count"] = (
    daytype_operating_period_candidates["candidate_operating_period_id"].map(term_counts)
)

daytype_operating_period_candidates

,daytype_ref,candidate_operating_period_id,daytype_ref_count,candidate_operating_period_count
0,VAR:DayType:T2026-BR-VS-M-P,VAR:OperatingPeriod:T2026-BR-VS-M-P,1,1
1,OYM:DayType:207805.474,OYM:OperatingPeriod:207805.474,1,0
2,OYM:DayType:226124.474,OYM:OperatingPeriod:226124.474,1,0
3,OYM:DayType:226125.474,OYM:OperatingPeriod:226125.474,1,0
4,OYM:DayType:235060.7010,OYM:OperatingPeriod:235060.7010,1,0
5,OYM:DayType:235061.7010,OYM:OperatingPeriod:235061.7010,1,0
6,OYM:DayType:235073.7010,OYM:OperatingPeriod:235073.7010,1,0
7,OYM:DayType:235074.7010,OYM:OperatingPeriod:235074.7010,1,0
8,OYM:DayType:236358.7010,OYM:OperatingPeriod:236358.7010,1,0
9,OYM:DayType:236359.7010,OYM:OperatingPeriod:236359.7010,1,0


## Comment

The ID-pattern check gives a mixed result.

For the `VAR` example, the matching OperatingPeriod exists.  
This means that some DayTypes can be linked to OperatingPeriods by replacing `DayType` with `OperatingPeriod`.

However, this does not work for the `OYM` examples.  
Their candidate OperatingPeriod IDs were not found in `_shared_data.xml`.

So Finland does not use one single calendar-linking structure for all DayTypes:




- VAR DayTypes → can be linked to OperatingPeriod by similar ID

- OYM DayTypes → no matching OperatingPeriod found by this simple rule

# Conclusion: NeTEx calendar structure for Finland

The NeTEx calendar structure in Finland is significantly more complex and inconsistent
than in Norway or Sweden, and could not be fully reconstructed for comparison.

The exploration revealed that different operators use different calendar structures
within the same NeTEx feed. Some DayTypes store weekday patterns directly, some
are linked to OperatingPeriods by a matching ID pattern (as seen for VAR), and
others have no clear link to any date information at all (as seen for OYM).

This inconsistency is likely a consequence of the mixed production process: some
operators submit NeTEx natively while others have their GTFS data automatically
converted by Fintraffic's VACO tool and the conversion might not produce a uniform
calendar structure. 

For this reason, the calendar-level MVD comparison is not performed for Finland.
Both formats contain calendar information, but the NeTEx calendar is too
mixed to reconstruct active dates in a consistent and reliable way across
all operators. 

This is itself a finding: `Finland's NeTEx feed does not yet have a consistent enough calendar structure to compare with GTFS the same way as Norway and Sweden`.

# Part 3: GTFS–NeTEx Comparison

## 3.1 Stop-level comparison for Finland

For Finland, the stop-level comparison uses all GTFS stops.

This is because NeTEx StopPlaces are not structured as parent stations here.  
They are closer to individual GTFS stops.

So the comparison is:

GTFS all stops  
vs  
NeTEx StopPlaces

I start with a coordinate-based comparison.

In [82]:
# Prepare GTFS and NeTEx stop tables for coordinate comparison

gtfs_stop_compare = gtfs_stops_all.copy()
netex_stop_compare = netex_stopplaces.copy()

# Create rounded coordinate keys
# Rounding avoids tiny decimal differences between the two formats
gtfs_stop_compare["coord_key"] = (
    gtfs_stop_compare["stop_lat"].round(6).astype(str)
    + "_"
    + gtfs_stop_compare["stop_lon"].round(6).astype(str)
)

netex_stop_compare["coord_key"] = (
    netex_stop_compare["latitude"].round(6).astype(str)
    + "_"
    + netex_stop_compare["longitude"].round(6).astype(str)
)

# Compare coordinate keys
gtfs_coord_keys = set(gtfs_stop_compare["coord_key"])
netex_coord_keys = set(netex_stop_compare["coord_key"])

stop_coord_key_comparison = pd.DataFrame({
    "metric": [
        "GTFS stop rows",
        "NeTEx StopPlace rows",
        "unique GTFS coordinate keys",
        "unique NeTEx coordinate keys",
        "GTFS coordinate keys found in NeTEx",
        "GTFS coordinate keys not found in NeTEx",
        "NeTEx coordinate keys found in GTFS",
        "NeTEx coordinate keys not found in GTFS"
    ],
    "value": [
        len(gtfs_stop_compare),
        len(netex_stop_compare),
        len(gtfs_coord_keys),
        len(netex_coord_keys),
        len(gtfs_coord_keys & netex_coord_keys),
        len(gtfs_coord_keys - netex_coord_keys),
        len(netex_coord_keys & gtfs_coord_keys),
        len(netex_coord_keys - gtfs_coord_keys)
    ]
})

stop_coord_key_comparison

,metric,value
0,GTFS stop rows,89547
1,NeTEx StopPlace rows,137227
2,unique GTFS coordinate keys,85133
3,unique NeTEx coordinate keys,109384
4,GTFS coordinate keys found in NeTEx,84248
5,GTFS coordinate keys not found in NeTEx,885
6,NeTEx coordinate keys found in GTFS,84248
7,NeTEx coordinate keys not found in GTFS,25136


In [83]:
# Row-level coordinate match check

gtfs_stop_compare["coord_found_in_netex"] = gtfs_stop_compare["coord_key"].isin(netex_coord_keys)
netex_stop_compare["coord_found_in_gtfs"] = netex_stop_compare["coord_key"].isin(gtfs_coord_keys)

stop_row_level_coord_coverage = pd.DataFrame({
    "metric": [
        "GTFS stops with coordinate found in NeTEx",
        "GTFS stops without coordinate found in NeTEx",
        "GTFS-side coordinate match rate",
        "NeTEx StopPlaces with coordinate found in GTFS",
        "NeTEx StopPlaces without coordinate found in GTFS",
        "NeTEx-side coordinate match rate"
    ],
    "value": [
        gtfs_stop_compare["coord_found_in_netex"].sum(),
        (~gtfs_stop_compare["coord_found_in_netex"]).sum(),
        round(gtfs_stop_compare["coord_found_in_netex"].mean() * 100, 2),
        netex_stop_compare["coord_found_in_gtfs"].sum(),
        (~netex_stop_compare["coord_found_in_gtfs"]).sum(),
        round(netex_stop_compare["coord_found_in_gtfs"].mean() * 100, 2)
    ]
})

stop_row_level_coord_coverage

,metric,value
0,GTFS stops with coordinate found in NeTEx,88640.00
1,GTFS stops without coordinate found in NeTEx,907.00
2,GTFS-side coordinate match rate,98.99
3,NeTEx StopPlaces with coordinate found in GTFS,95487.00
4,NeTEx StopPlaces without coordinate found in GTFS,41740.00
5,NeTEx-side coordinate match rate,69.58


## Comment

Most GTFS stops are found in NeTEx &rarr; GTFS-side match rate: 98.99%

So almost every GTFS stop has a matching NeTEx StopPlace by rounded coordinates.

But NeTEx has many extra StopPlaces &rarr; NeTEx-side match rate: 69.58%

This means many NeTEx StopPlaces do not have a matching GTFS coordinate. That fits the expected: the NeTEx feed is larger and may contain extra converted entries, parent-like entries, or additional operator data.

## Inspect GTFS stops not matched by coordinates




In [84]:
# Extract GTFS stops that were not matched by coordinate

gtfs_unmatched_stops = gtfs_stop_compare[
    ~gtfs_stop_compare["coord_found_in_netex"]
].copy()

gtfs_unmatched_stops[[
    "stop_id",
    "stop_name",
    "stop_lat",
    "stop_lon",
    "location_type",
    "parent_station"
]].head(20)

,stop_id,stop_name,stop_lat,stop_lon,location_type,parent_station
7745,308046,Rautatientori,60.170760,24.942680,0,300002.0
7746,308047,Rautatientori,60.170720,24.943240,0,300002.0
7747,308048,Rautatientori,60.171410,24.942660,0,300002.0
7748,308049,Rautatientori,60.171080,24.942830,0,300002.0
7749,308050,Rautatientori,60.171330,24.942770,0,300002.0
7750,308051,Rautatientori,60.170950,24.943080,0,300002.0
7751,308052,Rautatientori,60.171320,24.943020,0,300002.0
7752,308053,Rautatientori,60.170850,24.943330,0,300002.0
7753,308054,Rautatientori,60.171340,24.943290,0,300002.0
7754,308055,Elielinaukio,60.171203,24.939580,0,300001.0


In [85]:
# Summary of unmatched GTFS stops

gtfs_unmatched_summary = pd.DataFrame({
    "metric": [
        "unmatched GTFS stop rows",
        "unique unmatched stop names",
        "unique unmatched coordinate keys",
        "missing names",
        "missing latitudes",
        "missing longitudes"
    ],
    "value": [
        len(gtfs_unmatched_stops),
        gtfs_unmatched_stops["stop_name"].nunique(),
        gtfs_unmatched_stops["coord_key"].nunique(),
        gtfs_unmatched_stops["stop_name"].isna().sum(),
        gtfs_unmatched_stops["stop_lat"].isna().sum(),
        gtfs_unmatched_stops["stop_lon"].isna().sum()
    ]
})

gtfs_unmatched_summary

,metric,value
0,unmatched GTFS stop rows,907
1,unique unmatched stop names,360
2,unique unmatched coordinate keys,885
3,missing names,0
4,missing latitudes,0
5,missing longitudes,0


In [86]:
# Check location_type among unmatched GTFS stops

gtfs_unmatched_location_types = (
    gtfs_unmatched_stops["location_type"]
    .fillna("missing")
    .value_counts(dropna=False)
    .reset_index()
)

gtfs_unmatched_location_types.columns = ["location_type", "n_unmatched_stops"]

gtfs_unmatched_location_types

,location_type,n_unmatched_stops
0,0,907


## Comment

907 GTFS stops were not matched by rounded coordinates.

These stops are not missing names or coordinates.  
All of them are normal stop rows with `location_type = 0`.

So the unmatched rows are not caused by missing data.

They are probably individual stop/platform points where the GTFS and NeTEx coordinates are slightly different.



## Nearest-neighbour distance check for unmatched GTFS stops

Not all GTFS stops had an exact coordinate match in NeTEx. For the remaining
unmatched stops, I use a nearest-neighbour search to find the closest NeTEx
StopPlace and measure the distance in metres.

The search uses a `BallTree` with the haversine metric, which computes accurate
great-circle distances on the Earth's surface. This is more precise than the
degree-to-metre approximation used for Norway and Sweden, and is the correct
approach for geographic coordinates at Finland's latitude.

For each unmatched GTFS stop, the nearest NeTEx StopPlace is identified and the
distance to it is recorded. The second cell summarises the distance distribution
across multiple thresholds (5m, 10m, 25m, 50m, 100m) to understand how close
the unmatched stops are to their nearest NeTEx counterpart and whether a
proximity threshold could be used to recover additional matches.

In [87]:
# Prepare coordinates in radians for haversine distance
netex_coords_rad = np.radians(
    netex_stop_compare[["latitude", "longitude"]].values
)

gtfs_unmatched_coords_rad = np.radians(
    gtfs_unmatched_stops[["stop_lat", "stop_lon"]].values
)

# Build nearest-neighbour tree from NeTEx coordinates
tree = BallTree(netex_coords_rad, metric="haversine")

# Find nearest NeTEx StopPlace for each unmatched GTFS stop
distances_rad, indices = tree.query(gtfs_unmatched_coords_rad, k=1)

# Convert distance from radians to meters
earth_radius_m = 6371000
distances_m = distances_rad[:, 0] * earth_radius_m
nearest_indices = indices[:, 0]

# Create nearest-neighbour result table
gtfs_unmatched_nearest = gtfs_unmatched_stops.reset_index(drop=True).copy()

nearest_netex = netex_stop_compare.iloc[nearest_indices].reset_index(drop=True)

gtfs_unmatched_nearest["nearest_netex_stopplace_id"] = nearest_netex["stopplace_id"]
gtfs_unmatched_nearest["nearest_netex_name"] = nearest_netex["name"]
gtfs_unmatched_nearest["nearest_netex_latitude"] = nearest_netex["latitude"]
gtfs_unmatched_nearest["nearest_netex_longitude"] = nearest_netex["longitude"]
gtfs_unmatched_nearest["distance_to_nearest_netex_m"] = distances_m

gtfs_unmatched_nearest[[
    "stop_id",
    "stop_name",
    "stop_lat",
    "stop_lon",
    "nearest_netex_stopplace_id",
    "nearest_netex_name",
    "nearest_netex_latitude",
    "nearest_netex_longitude",
    "distance_to_nearest_netex_m"
]].head(20)

,stop_id,stop_name,stop_lat,stop_lon,nearest_netex_stopplace_id,nearest_netex_name,nearest_netex_latitude,nearest_netex_longitude,distance_to_nearest_netex_m
0,308046,Rautatientori,60.170760,24.942680,FSR:StopPlace:300002,Rautatientori,60.171283,24.942572,58.460927
1,308047,Rautatientori,60.170720,24.943240,FSR:StopPlace:300002,Rautatientori,60.171283,24.942572,72.692372
2,308048,Rautatientori,60.171410,24.942660,FSR:StopPlace:300002,Rautatientori,60.171283,24.942572,14.936992
3,308049,Rautatientori,60.171080,24.942830,FSR:StopPlace:300002,Rautatientori,60.171283,24.942572,26.704862
4,308050,Rautatientori,60.171330,24.942770,FSR:StopPlace:300002,Rautatientori,60.171283,24.942572,12.134350
5,308051,Rautatientori,60.170950,24.943080,FSR:StopPlace:300002,Rautatientori,60.171283,24.942572,46.481433
6,308052,Rautatientori,60.171320,24.943020,FSR:StopPlace:300002,Rautatientori,60.171283,24.942572,25.117807
7,308053,Rautatientori,60.170850,24.943330,FSR:StopPlace:300002,Rautatientori,60.171283,24.942572,63.842443
8,308054,Rautatientori,60.171340,24.943290,FSR:StopPlace:300002,Rautatientori,60.171283,24.942572,40.214676
9,308055,Elielinaukio,60.171203,24.939580,FSR:StopPlace:300001,Elielinaukio,60.171808,24.939488,67.465097


In [88]:
# Summarise nearest-neighbour distances

nearest_distance_summary = pd.DataFrame({
    "metric": [
        "unmatched GTFS stops checked",
        "minimum distance to nearest NeTEx stop (m)",
        "median distance to nearest NeTEx stop (m)",
        "mean distance to nearest NeTEx stop (m)",
        "maximum distance to nearest NeTEx stop (m)",
        "stops within 5 m",
        "stops within 10 m",
        "stops within 25 m",
        "stops within 50 m",
        "stops within 100 m"
    ],
    "value": [
        len(gtfs_unmatched_nearest),
        round(gtfs_unmatched_nearest["distance_to_nearest_netex_m"].min(), 2),
        round(gtfs_unmatched_nearest["distance_to_nearest_netex_m"].median(), 2),
        round(gtfs_unmatched_nearest["distance_to_nearest_netex_m"].mean(), 2),
        round(gtfs_unmatched_nearest["distance_to_nearest_netex_m"].max(), 2),
        (gtfs_unmatched_nearest["distance_to_nearest_netex_m"] <= 5).sum(),
        (gtfs_unmatched_nearest["distance_to_nearest_netex_m"] <= 10).sum(),
        (gtfs_unmatched_nearest["distance_to_nearest_netex_m"] <= 25).sum(),
        (gtfs_unmatched_nearest["distance_to_nearest_netex_m"] <= 50).sum(),
        (gtfs_unmatched_nearest["distance_to_nearest_netex_m"] <= 100).sum()
    ]
})

nearest_distance_summary

,metric,value
0,unmatched GTFS stops checked,907.00
1,minimum distance to nearest NeTEx stop (m),0.05
2,median distance to nearest NeTEx stop (m),24.63
3,mean distance to nearest NeTEx stop (m),41.49
4,maximum distance to nearest NeTEx stop (m),331.20
5,stops within 5 m,113.00
6,stops within 10 m,228.00
7,stops within 25 m,457.00
8,stops within 50 m,655.00
9,stops within 100 m,815.00


## Comment

The nearest-neighbour check shows that most unmatched GTFS stops are still close to a NeTEx StopPlace.

The median distance is 24.63 metres.  
815 out of 907 unmatched GTFS stops are within 100 metres of a NeTEx StopPlace.

The examples also show similar names, such as `Rautatientori` and `Elielinaukio`.

This means the unmatched GTFS stops are probably not truly missing.  
Most of them are likely caused by small coordinate differences or by slightly different stop granularity.


## Check names for nearest NeTEx matches

The nearest-neighbour check showed that most unmatched GTFS stops are close to a NeTEx StopPlace.

Now I check whether the closest NeTEx StopPlace also has the same stop name.

In [89]:
# Normalise stop names for simple comparison

def normalize_name(value):
    # Handle missing values
    if pd.isna(value):
        return None
    
    # Convert to string and lowercase
    value = str(value).strip().casefold()
    
    # Normalise unicode characters
    value = unicodedata.normalize("NFKC", value)
    
    # Replace multiple spaces with one space
    value = re.sub(r"\s+", " ", value)
    
    return value

# Add normalised GTFS and NeTEx names

gtfs_unmatched_nearest["gtfs_name_norm"] = gtfs_unmatched_nearest["stop_name"].apply(normalize_name)
gtfs_unmatched_nearest["netex_name_norm"] = gtfs_unmatched_nearest["nearest_netex_name"].apply(normalize_name)

# Check exact name match after normalisation

gtfs_unmatched_nearest["nearest_name_match"] = (
    gtfs_unmatched_nearest["gtfs_name_norm"] == gtfs_unmatched_nearest["netex_name_norm"]
)

gtfs_unmatched_nearest[[
    "stop_id",
    "stop_name",
    "nearest_netex_stopplace_id",
    "nearest_netex_name",
    "distance_to_nearest_netex_m",
    "nearest_name_match"
]].head(20)

,stop_id,stop_name,nearest_netex_stopplace_id,nearest_netex_name,distance_to_nearest_netex_m,nearest_name_match
0,308046,Rautatientori,FSR:StopPlace:300002,Rautatientori,58.460927,True
1,308047,Rautatientori,FSR:StopPlace:300002,Rautatientori,72.692372,True
2,308048,Rautatientori,FSR:StopPlace:300002,Rautatientori,14.936992,True
3,308049,Rautatientori,FSR:StopPlace:300002,Rautatientori,26.704862,True
4,308050,Rautatientori,FSR:StopPlace:300002,Rautatientori,12.134350,True
5,308051,Rautatientori,FSR:StopPlace:300002,Rautatientori,46.481433,True
6,308052,Rautatientori,FSR:StopPlace:300002,Rautatientori,25.117807,True
7,308053,Rautatientori,FSR:StopPlace:300002,Rautatientori,63.842443,True
8,308054,Rautatientori,FSR:StopPlace:300002,Rautatientori,40.214676,True
9,308055,Elielinaukio,FSR:StopPlace:300001,Elielinaukio,67.465097,True


In [90]:
# Summarise nearest matches by distance and name

nearest_name_distance_summary = pd.DataFrame({
    "metric": [
        "unmatched GTFS stops checked",
        "nearest NeTEx stop has same name",
        "nearest NeTEx stop has different name",
        "same name and within 25 m",
        "same name and within 50 m",
        "same name and within 100 m",
        "different name but within 100 m",
        "farther than 100 m"
    ],
    "value": [
        len(gtfs_unmatched_nearest),
        gtfs_unmatched_nearest["nearest_name_match"].sum(),
        (~gtfs_unmatched_nearest["nearest_name_match"]).sum(),
        ((gtfs_unmatched_nearest["nearest_name_match"]) &
         (gtfs_unmatched_nearest["distance_to_nearest_netex_m"] <= 25)).sum(),
        ((gtfs_unmatched_nearest["nearest_name_match"]) &
         (gtfs_unmatched_nearest["distance_to_nearest_netex_m"] <= 50)).sum(),
        ((gtfs_unmatched_nearest["nearest_name_match"]) &
         (gtfs_unmatched_nearest["distance_to_nearest_netex_m"] <= 100)).sum(),
        ((~gtfs_unmatched_nearest["nearest_name_match"]) &
         (gtfs_unmatched_nearest["distance_to_nearest_netex_m"] <= 100)).sum(),
        (gtfs_unmatched_nearest["distance_to_nearest_netex_m"] > 100).sum()
    ]
})

nearest_name_distance_summary

,metric,value
0,unmatched GTFS stops checked,907
1,nearest NeTEx stop has same name,374
2,nearest NeTEx stop has different name,533
3,same name and within 25 m,157
4,same name and within 50 m,243
5,same name and within 100 m,315
6,different name but within 100 m,500
7,farther than 100 m,92


In [91]:
# Inspect cases farther than 100 m

gtfs_unmatched_nearest[
    gtfs_unmatched_nearest["distance_to_nearest_netex_m"] > 100
][[
    "stop_id",
    "stop_name",
    "stop_lat",
    "stop_lon",
    "nearest_netex_stopplace_id",
    "nearest_netex_name",
    "nearest_netex_latitude",
    "nearest_netex_longitude",
    "distance_to_nearest_netex_m",
    "nearest_name_match"
]].sort_values("distance_to_nearest_netex_m").head(30)

,stop_id,stop_name,stop_lat,stop_lon,nearest_netex_stopplace_id,nearest_netex_name,nearest_netex_latitude,nearest_netex_longitude,distance_to_nearest_netex_m,nearest_name_match
25,308073,Helsinki,60.172987,24.941576,FSR:StopPlace:315477,Helsinki,60.172097,24.941249,100.602461,True
21,308069,Helsinki,60.172720,24.939911,FSR:StopPlace:315477,Helsinki,60.172097,24.941249,101.366410,True
169,308223,Mellunmäki (M),60.237586,25.110548,FSR:StopPlace:441461,Mellunmäki,60.238248,25.109276,101.725769,False
24,308072,Helsinki,60.172725,24.939595,FSR:StopPlace:300001,Elielinaukio,60.171808,24.939488,102.137337,False
664,316454,Suonenjoki,62.622818,27.125121,FSR:StopPlace:315780,Suonenjoki,62.623707,27.124495,103.941071,True
160,308214,Kontula (M),60.235970,25.080871,FSR:StopPlace:300006,Kontula,60.236413,25.082534,104.179125,False
603,316303,Oulu,65.010762,25.483431,FSR:StopPlace:330943,Matkakeskus P,65.011043,25.481267,106.361008,False
512,316076,Kokemäki,61.253760,22.305133,FSR:StopPlace:315562,Kokemäki,61.254199,22.303350,107.100769,True
831,388437,Kokemäki,61.253714,22.305082,FSR:StopPlace:315562,Kokemäki,61.254199,22.303350,107.204678,True
581,316230,Lappeenranta,61.046906,28.192817,FSR:StopPlace:329185,Ratakatu 18 keskustasta,61.047765,28.191876,108.140692,False


## Comment

The nearest-neighbour check gives extra support for the stop-level comparison.

Most unmatched GTFS stops are still close to a NeTEx StopPlace.  
815 out of 907 are within 100 metres.

374 of the nearest NeTEx stops have exactly the same name.  
533 have a different name, but some of these differences are only naming variants, for example station suffixes or slightly longer NeTEx names.

 This check shows that many of the remaining unmatched GTFS stops are not completely missing, but are probably caused by coordinate shifts or naming differences.

## Final stop-level comparison summary

The main stop-level comparison for Finland remains the rounded coordinate matching.

The nearest-neighbour check is used only as supporting evidence and it helps explain the GTFS stops that were not matched by exact rounded coordinates.  


In [92]:
# Final stop-level comparison summary for Finland

final_stop_level_summary = pd.DataFrame({
    "metric": [
        "GTFS stop rows",
        "NeTEx StopPlace rows",
        "GTFS stops matched by rounded coordinates",
        "GTFS stops not matched by rounded coordinates",
        "GTFS-side rounded-coordinate match rate",
        "NeTEx StopPlaces matched by rounded coordinates",
        "NeTEx StopPlaces not matched by rounded coordinates",
        "NeTEx-side rounded-coordinate match rate",
        "unmatched GTFS stops checked with nearest-neighbour method",
        "of unmatched GTFS stops, within 100 m of a NeTEx StopPlace",
        "of unmatched GTFS stops, farther than 100 m"
    ],
    "value": [
        len(gtfs_stop_compare),
        len(netex_stop_compare),
        gtfs_stop_compare["coord_found_in_netex"].sum(),
        (~gtfs_stop_compare["coord_found_in_netex"]).sum(),
        round(gtfs_stop_compare["coord_found_in_netex"].mean() * 100, 2),
        netex_stop_compare["coord_found_in_gtfs"].sum(),
        (~netex_stop_compare["coord_found_in_gtfs"]).sum(),
        round(netex_stop_compare["coord_found_in_gtfs"].mean() * 100, 2),
        len(gtfs_unmatched_nearest),
        (gtfs_unmatched_nearest["distance_to_nearest_netex_m"] <= 100).sum(),
        (gtfs_unmatched_nearest["distance_to_nearest_netex_m"] > 100).sum()
    ]
})

final_stop_level_summary

,metric,value
0,GTFS stop rows,89547.00
1,NeTEx StopPlace rows,137227.00
2,GTFS stops matched by rounded coordinates,88640.00
3,GTFS stops not matched by rounded coordinates,907.00
4,GTFS-side rounded-coordinate match rate,98.99
5,NeTEx StopPlaces matched by rounded coordinates,95487.00
6,NeTEx StopPlaces not matched by rounded coordi...,41740.00
7,NeTEx-side rounded-coordinate match rate,69.58
8,unmatched GTFS stops checked with nearest-neig...,907.00
9,"of unmatched GTFS stops, within 100 m of a NeT...",815.00


## Comment

The final stop-level comparison shows a strong GTFS-side result.

88,640 out of 89,547 GTFS stops are matched by rounded coordinates.  
This gives a GTFS-side match rate of 98.99%.

The NeTEx-side match rate is lower, at 69.58%, because NeTEx has many more StopPlace rows than GTFS has stop rows.

The nearest-neighbour check supports the result.  
Out of the 907 GTFS stops not matched by rounded coordinates, 815 are still within 100 metres of a NeTEx StopPlace.

So the stop-level comparison is strongly supported for Finland from the GTFS side.  
The lower NeTEx-side rate is mainly caused by the larger NeTEx stop table and different stop granularity.

## 3.2 Line-level comparison for Finland

For the line-level comparison, I compare public-facing line labels.

On the GTFS side, I use the prepared GTFS line label table.  
On the NeTEx side, I use the deduplicated NeTEx PublicCode table.


In [93]:
# Prepare label sets for comparison

gtfs_line_label_set = set(
    gtfs_lines["gtfs_line_label"]
    .dropna()
    .astype("string")
    .str.strip()
)

netex_line_label_set = set(
    netex_line_labels["netex_line_label"]
    .dropna()
    .astype("string")
    .str.strip()
)

matched_line_labels = gtfs_line_label_set & netex_line_label_set
gtfs_only_line_labels = gtfs_line_label_set - netex_line_label_set
netex_only_line_labels = netex_line_label_set - gtfs_line_label_set

line_label_comparison_summary = pd.DataFrame({
    "metric": [
        "unique GTFS line labels",
        "unique NeTEx line labels",
        "matched line labels",
        "GTFS-only line labels",
        "NeTEx-only line labels",
        "GTFS-side line-label match rate",
        "NeTEx-side line-label match rate"
    ],
    "value": [
        len(gtfs_line_label_set),
        len(netex_line_label_set),
        len(matched_line_labels),
        len(gtfs_only_line_labels),
        len(netex_only_line_labels),
        round(len(matched_line_labels) / len(gtfs_line_label_set) * 100, 2),
        round(len(matched_line_labels) / len(netex_line_label_set) * 100, 2)
    ]
})

line_label_comparison_summary

,metric,value
0,unique GTFS line labels,3512.00
1,unique NeTEx line labels,1875.00
2,matched line labels,1865.00
3,GTFS-only line labels,1647.00
4,NeTEx-only line labels,10.00
5,GTFS-side line-label match rate,53.10
6,NeTEx-side line-label match rate,99.47


In [94]:
print("GTFS-only line labels:")
display(pd.DataFrame({
    "gtfs_only_label": sorted(gtfs_only_line_labels)[:30]
}))

print("NeTEx-only line labels:")
display(pd.DataFrame({
    "netex_only_label": sorted(netex_only_line_labels)[:30]
}))

print("Example matched line labels:")
display(pd.DataFrame({
    "matched_label": sorted(matched_line_labels)[:30]
}))

GTFS-only line labels:


,gtfs_only_label
0,166K
1,181B Kortesjärvi - Purmojärvi - Evijärvi
2,182Y
3,24S
4,354
5,570N
6,Ahvensalmi - Enonkoski
7,Ahvensalmi - Savonlinna
8,Aitoo - Luopioinen - Kyynärö
9,Aitoo - Sahalahti - Tampere


NeTEx-only line labels:


,netex_only_label
0,350A
1,350B
2,351B
3,352A
4,352B
5,353A
6,353B
7,354B
8,355A
9,3X


Example matched line labels:


,matched_label
0,1
1,10
2,100
3,1001
4,1003
5,101
6,1011
7,1012
8,102
9,102N


## Comment

The line-level comparison gives a strongly asymmetric result, which could be 
explained by how GTFS line labels were constructed for Finland:

- **NeTEx-side match rate: 99.47%**: almost every NeTEx line label has a
  matching GTFS label, meaning NeTEx coverage is essentially complete relative
  to GTFS
- **GTFS-side match rate: 53.10%**: about half of GTFS line labels have no
  matching NeTEx label, but this is largely a labelling issue rather than
  genuinely missing lines
- Many GTFS-only labels are full route names like "Alavus - Kattelus" or
  "Ahvensalmi - Savonlinna", used as fallback labels when `route_short_name`
  was missing. These long names naturally do not match NeTEx public codes
- Only 10 NeTEx-only labels exist, all alphanumeric codes like "350A" or "351B",
  likely variant lines that NeTEx models separately but GTFS groups under a
  single label
- Matched labels are clean short numeric codes, confirming that properly coded
  lines are consistent across both formats

Overall, the line-level comparison works for Finland for lines with a proper
public code. The lower GTFS-side rate reflects a data quality issue in the GTFS
feed rather than a fundamental incompatibility between the two formats.

## Inspect unmatched line labels

The first line-level comparison showed a strong NeTEx-side match, but a lower GTFS-side match.

I inspect the unmatched labels more closely because some GTFS labels may not be real public line codes.  
In Finland, when `route_short_name` was missing, the route long name was used as a fallback label.

These fallback labels are often full route descriptions, such as place-to-place names.  
They are not expected to match NeTEx `PublicCode` values.



In [95]:
# Inspect GTFS-only line labels more closely

gtfs_only_details = gtfs_lines[
    gtfs_lines["gtfs_line_label"].isin(gtfs_only_line_labels)
].copy()

gtfs_only_details["gtfs_line_label"] = (
    gtfs_only_details["gtfs_line_label"]
    .astype("string")
    .str.strip()
)

gtfs_only_details["label_length"] = gtfs_only_details["gtfs_line_label"].str.len()

gtfs_only_details["contains_space"] = gtfs_only_details["gtfs_line_label"].str.contains(" ", regex=False)
gtfs_only_details["contains_dash"] = gtfs_only_details["gtfs_line_label"].str.contains("-", regex=False)

gtfs_only_details["looks_like_short_code"] = (
    gtfs_only_details["gtfs_line_label"]
    .str.match(r"^[A-Za-z0-9]{1,6}$", na=False)
)

gtfs_only_label_shape_summary = pd.DataFrame({
    "metric": [
        "GTFS-only line labels",
        "GTFS-only labels that look like short codes",
        "GTFS-only labels longer than 6 characters",
        "GTFS-only labels containing spaces",
        "GTFS-only labels containing dashes"
    ],
    "value": [
        len(gtfs_only_details),
        gtfs_only_details["looks_like_short_code"].sum(),
        (gtfs_only_details["label_length"] > 6).sum(),
        gtfs_only_details["contains_space"].sum(),
        gtfs_only_details["contains_dash"].sum()
    ]
})

gtfs_only_label_shape_summary

,metric,value
0,GTFS-only line labels,1647
1,GTFS-only labels that look like short codes,5
2,GTFS-only labels longer than 6 characters,1642
3,GTFS-only labels containing spaces,1606
4,GTFS-only labels containing dashes,1640


In [96]:
# Show examples of GTFS-only labels that look like route descriptions

gtfs_only_details[
    gtfs_only_details["label_length"] > 6
][[
    "gtfs_line_label",
    "n_route_rows",
    "route_types",
    "example_route_id",
    "example_route_long_name"
]].head(30)

,gtfs_line_label,n_route_rows,route_types,example_route_id,example_route_long_name
177,181B Kortesjärvi - Purmojärvi - Evijärvi,1,[704],145577,181B
1275,Ahvensalmi - Enonkoski,1,[3],24177,Ahvensalmi - Enonkoski
1276,Ahvensalmi - Savonlinna,1,[3],311115,Ahvensalmi - Savonlinna
1277,Aitoo - Luopioinen - Kyynärö,1,[3],152640,Aitoo - Luopioinen - Kyynärö
1278,Aitoo - Sahalahti - Tampere,1,[3],23828,Aitoo - Sahalahti - Tampere
1279,Akaa rautatieasema (Toijala) - Urjalan linja-a...,1,[3],347239,Akaa rautatieasema (Toijala) - Urjalan linja-a...
1280,Akaa rautatieasema (Toijala) - Valkeakosken li...,1,[3],347242,Akaa rautatieasema (Toijala) - Valkeakosken li...
1281,Alastaipale - Virrat,1,[3],26071,Alastaipale - Virrat
1282,Alastaipale E - Virrat MH,1,[3],347206,Alastaipale E - Virrat MH
1283,Alavus - Hietaranta,1,[3],141372,Alavus - Hietaranta


In [97]:
# Inspect NeTEx-only labels

netex_only_details = netex_line_labels[
    netex_line_labels["netex_line_label"].isin(netex_only_line_labels)
].copy()

netex_only_details[[
    "netex_line_label",
    "n_line_rows",
    "transport_modes",
    "example_line_id",
    "example_name"
]]

,netex_line_label,n_line_rows,transport_modes,example_line_id,example_name
424,350A,1,[bus],FSR:Line:128762,Lapväärtti - Närpiö - Vaasa
425,350B,1,[bus],FSR:Line:128760,Vaasa - Närpiö - Lapväärtti
427,351B,1,[bus],FSR:Line:115351,Vaasa - Maalahti - Lapväärtti
429,352A,1,[bus],FSR:Line:162279,Dagsmark - Närpiö - Vaasa
430,352B,1,[bus],FSR:Line:162280,Vaasa - Närpiö - Dagsmark
432,353A,1,[bus],FSR:Line:128766,Kaskinen - Maalahti - Vaasa
433,353B,1,[bus],FSR:Line:128765,Vaasa - Maalahti - Kaskinen
434,354B,1,[bus],FSR:Line:115352,Vaasa - Närpiö - Kaskinen
436,355A,1,[bus],FSR:Line:162281,Närpiö - Maalahti - Vaasa
484,3X,2,[bus],FSR:Line:314129,Eteläranta - Lasipalatsi


## Comment

The additional check confirms that most GTFS-only labels are not short public line codes.

Out of 1,647 GTFS-only labels, only 5 look like short codes.  
Almost all GTFS-only labels are longer than 6 characters, and most contain spaces and dashes.

The examples show that these labels are mostly full route descriptions, such as place-to-place names.

This means that the lower GTFS-side match rate is mainly caused by the way GTFS labels were constructed when `route_short_name` was missing.

The NeTEx-only labels are only 10 cases. They are short bus codes such as `350A`, `350B`, and `3X`.

Overall, the line-level comparison works very well for proper public line codes. Almost all NeTEx PublicCodes are present in GTFS, and the unmatched GTFS labels are mostly descriptive fallback labels rather than comparable public codes.

# Conclusion: line-level comparison for Finland

The line-level comparison shows a strongly asymmetric result. Almost all NeTEx
line labels (99.47%) are found in GTFS, but only 53.10% of GTFS labels match
a NeTEx public code. The additional inspection confirmed that most GTFS-only
labels are not proper short public codes, they are long route names used as
fallback labels when `route_short_name` was missing. When only short public codes
are considered, the two formats are highly consistent. The line-level MVD is
confirmed for Finland for lines with a proper public code.

## 3.3 Calendar comparison for Finland

The calendar-level comparison is not performed for Finland. As documented in
Part 2, the Finnish NeTEx calendar structure is inconsistent across operators
and could not be reconstructed in a reliable way. The NeTEx feed does not yet
have a consistent enough calendar structure to compare with GTFS the same way
as Norway and Sweden. This shows that NeTEx is not used in a fully consistent way in Finland yet.